# YOLO-World: Data Pipeline and Experiments

**Project:** YOLO-World: Real-Time Open-Vocabulary Object Detection (Cheng et al., CVPR 2024, arXiv:2401.17270)
**Group:** S. Mujtaba Kazmi (36397) · Syed Mizfar Hussain (36251) · Ayaan Hussain (36223)

This notebook continues our submitted notebook *YOLO World Data Pipeline.ipynb*. Sections 1 to 7 are the same data pipeline (only the custom dataset part and some settings are new). Sections 3b and 8 to 13 carry out the **four next steps** from the end of our report:

1. choose the custom fine tuning dataset and repeat the EDA on it (**Section 3b**),
2. measure zero-shot AP on the COCO and LVIS subsets (**Section 9**),
3. run the prompt-sensitivity study (**Section 10**) and the S/M/L speed study (**Section 11**),
4. fine-tune YOLO-World and compare it with a closed-set YOLOv8 (**Section 12**).

## What this notebook does

| Section | What happens |
|---|---|
| 1. Setup | Install the libraries, fix the random seed, find the GPU, all settings in `CFG` |
| 2. COCO and LVIS data | Download a small part of COCO val2017 (and LVIS v1 val) and keep it in one simple format |
| 2c. EDA | Class counts, image sizes, box sizes and data checks for COCO |
| 3. Custom data | Download our custom dataset (African Wildlife) and read it in YOLO format |
| **3b. Custom data EDA** | **Next step 1:** the same EDA and data checks on the custom dataset |
| 4. Image preprocessing | Letterbox resize, pixels scaled to 0 to 1, and training augmentations |
| 4b. Text sampling | Give every training image its own list of class names, like YOLO-World does |
| 5. PyTorch Dataset | One `Dataset` class and a `DataLoader` for all data |
| 6. Text preprocessing | Prompts, CLIP tokens, saved text embeddings, check with the official class list |
| 7. Checks | Pictures, automatic checks, and a test run of the pretrained YOLO-World |
| **8. Experiment setup** | Shared code: official COCO and LVIS evaluation, timing, result tables |
| **9. Zero-shot evaluation** | **Next step 2:** COCO mAP@[.5:.95], AP50, AP75, AP small/medium/large, AP per class; LVIS AP with rare/common/frequent groups |
| **10. Prompt sensitivity** | **Next step 3a:** how the wording of the prompts changes AP on COCO and on the custom data |
| **11. S/M/L study** | **Next step 3b:** accuracy, latency, FPS, memory and size of the S, M and L models |
| **11b. Reproduction** | Our zero-shot COCO/LVIS numbers and speed next to the numbers reported in the paper and by Ultralytics |
| **12. Fine tuning** | **Next step 4 (additional experiment):** fine-tune YOLO-World-S and train a closed-set YOLOv8-S on the same data, then compare them; trained weights are saved to `weights/` |
| **13. Results summary** | All result tables are saved in `data/results/` and the key findings are printed |

**Deliverable 3 package.** This notebook is the experiment record. The same folder also has `train.py`, `evaluate.py`, `infer.py`, the Streamlit app `app.py`, `requirements.txt`, and a `README.md` with the setup and launch steps and the full **code attribution** table (what we wrote, what we adapted, what we reused).

**How to run:** in Google Colab choose *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The full run downloads about 1.5 GB (annotations, 500 COCO and 500 LVIS images, the custom dataset and the model weights). Most of the time goes to the two training runs in Section 12. To check that every cell works before the full run, set `CFG["QUICK_TEST"] = True` in Section 1: it uses a few images and one training epoch, and its numbers are **not** results.

**All numbers come from running this notebook.** The text cells explain the method; the result tables and the findings in Section 13 are printed by the code.

## Remarks

The official YOLO-World code (`AILab-CVC/YOLO-World` on GitHub) is built on MMYOLO, MMDetection and MMCV. It needs many GPUs and very big training sets (Objects365, GoldG, CC3M). We only have one free GPU, so we made these changes:

1. **No MMCV or MMDetection.** We wrote the same steps (letterbox, mosaic, random zoom and shift, colour change, flip, divide by 255) with NumPy, OpenCV and PyTorch.
2. **Only a small part of the data.** We read the full annotation file but download only 500 random COCO val2017 images.
3. **One simple record for every dataset.** Each image is a small dictionary with `image_path`, `boxes` and `labels`. COCO, LVIS and our custom data all use it.
4. **Text embeddings are made once and saved.** We use the CLIP text model `openai/clip-vit-base-patch32`, the same one YOLO-World uses. Making them once before detection is the "prompt then detect" idea of the paper.
5. **No pre-training.** We use the released YOLO-World weights through the Ultralytics library (Section 7c). In Section 7c we check that its text embeddings are the same as ours.
6. **Same settings as the official config.** Our augmentation numbers (zoom 0.5 to 1.5, grey value 114, max box aspect ratio 100, HSV gains, flip 0.5, divide by 255) are taken from the official file `configs/finetune_coco/yolo_world_v2_s_vlpan_bn_2e-4_80e_8gpus_mask-refine_finetune_coco.py`.

## 1. Setup

Run the install cell once. The `ultralytics` library has a YOLO-World model with the authors' weights, and it is much easier to install than MMYOLO.

In [ ]:
# %pip installs the packages into the same Python that runs this notebook.
%pip install -q ultralytics pycocotools transformers opencv-python-headless matplotlib tqdm pyyaml requests pandas
# Official LVIS evaluation code (Section 9b). --no-deps: everything it needs is already installed above.
%pip install -q --no-deps lvis

# Only needed if we download the custom dataset from Roboflow:
# %pip install -q roboflow

In [ ]:
# Basic Python tools
import os, json, random, time, zipfile, math
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor   # to download many images at the same time

# Other libraries
import numpy as np
import cv2                      # read, resize and recolour images
import requests                 # download files from the internet
import yaml                     # read the data.yaml file of a YOLO dataset
import torch
from PIL import Image           # quick way to get the size of an image
from tqdm.auto import tqdm      # progress bars
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Random seed
# With a fixed seed we get the same random images and the same augmentations every time we run.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Use the GPU if there is one, otherwise the CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU found")

# All settings in one place
CFG = dict(
    ROOT        = Path("./data"),   # all files are saved in this folder
    IMG_SIZE    = 640,              # YOLO-World input size (640 x 640)
    COCO_SUBSET = 500,              # number of COCO val2017 images we use
    LVIS_SUBSET = 500,              # number of LVIS images (only if USE_LVIS is True)
    USE_LVIS    = True,             # LVIS zero-shot test in Section 9b (set False to skip it)
    BATCH_SIZE  = 8,
    # On Windows, extra loader workers give an error inside a notebook, so we use 0 there.
    # On Colab or Kaggle (Linux) we can use 2.
    NUM_WORKERS = 0 if os.name == "nt" else 2,
    PAD_VALUE   = 114,              # grey colour for padding, the normal value in YOLO
    # Settings for the experiments (Sections 8 to 13)
    CONF_EVAL   = 0.001,            # very low score limit for mAP, the normal COCO evaluation setting
    MAX_DET     = 300,              # at most 300 boxes per image (LVIS rule; COCO uses the best 100)
    CONF_SPEED  = 0.25,             # normal score limit for the speed test, like a real application
    SPEED_IMAGES = 100,             # number of images timed in the speed test
    MODEL_SIZES = ["s", "m", "l"],  # YOLO-World sizes for the S/M/L study
    EPOCHS      = 30,               # training epochs, the same for YOLO-World and YOLOv8
    TRAIN_BATCH = 16,
    QUICK_TEST  = False,            # True = tiny run only to test the code. The numbers are NOT results.
)
if CFG["QUICK_TEST"]:
    CFG.update(COCO_SUBSET=16, LVIS_SUBSET=8, SPEED_IMAGES=5, MODEL_SIZES=["s"], EPOCHS=1)
CFG["ROOT"].mkdir(parents=True, exist_ok=True)
CFG

## 2. Test data: COCO val2017

**One simple record for every image.** Every dataset is turned into a list of dictionaries like this:

```python
{"image_path": str,                   # where the image file is
 "boxes": np.ndarray (N, 4) float32,  # boxes in pixels: [x1, y1, x2, y2] (left, top, right, bottom)
 "labels": np.ndarray (N,) int64}     # class number of each box (position in the class_names list)
```

COCO saves a box as `[x, y, width, height]`, and its class ids go from 1 to 90 with gaps. We change both. Boxes marked as *crowd* (`iscrowd=1`) are removed, because one crowd box covers a group of objects and not one object.

In [ ]:
def download_file(url, dest, chunk=1 << 20):
    """Download a file from url and save it at dest.
    If the file is already there, we skip the download, so running the notebook again is fast.
    We first save to a ".part" file, so a half finished download is never used by mistake."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        # Show a progress bar only for big files like the annotation zip, not for each image
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                        desc=dest.name, disable=total < 5_000_000) as bar:
            for part in r.iter_content(chunk):
                f.write(part); bar.update(len(part))
    tmp.rename(dest)
    return dest


def clean_class_name(name: str) -> str:
    """Make a class name look like normal words, for example "alarm_clock" becomes "alarm clock".
    The CLIP text model works best with plain words."""
    return name.replace("_", " ").strip().lower()


def load_coco_style_subset(ann_json, img_dir, n_images, seed=SEED, min_box_side=2.0, workers=16):
    """Read a COCO style annotation file (COCO or LVIS) and load a random part of it.
    We download only n_images images, each one from its own coco_url,
    so we do not need the big 1 GB zip file. The result uses our simple record format."""
    with open(ann_json) as f:
        coco = json.load(f)

    # Step 1: COCO class ids have gaps (1 to 90). We give each class a new number from 0 to 79.
    cats = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_idx = {c["id"]: i for i, c in enumerate(cats)}
    class_names = [clean_class_name(c["name"]) for c in cats]

    # Step 2: Group the boxes by image and change [x, y, w, h] into [x1, y1, x2, y2].
    # We skip crowd boxes and tiny boxes (a side smaller than min_box_side pixels).
    per_image = defaultdict(list)
    for a in coco["annotations"]:
        if a.get("iscrowd", 0):
            continue
        x, y, w, h = a["bbox"]
        if w < min_box_side or h < min_box_side:
            continue
        per_image[a["image_id"]].append([x, y, x + w, y + h, cat_id_to_idx[a["category_id"]]])

    # Step 3: Pick random images that have at least one box.
    # Because of the fixed seed we get the same images every time.
    info = {im["id"]: im for im in coco["images"]}
    ids = sorted(per_image)
    random.Random(seed).shuffle(ids)
    ids = ids[:n_images]

    # Step 4: Download the chosen images. We use many threads, so many images download together.
    img_dir = Path(img_dir); img_dir.mkdir(parents=True, exist_ok=True)
    def fetch(i):
        url = info[i]["coco_url"]
        try:
            return i, download_file(url, img_dir / url.split("/")[-1])
        except Exception as e:           # if one image fails, we skip it
            return i, None
    with ThreadPoolExecutor(workers) as ex:
        fetched = list(tqdm(ex.map(fetch, ids), total=len(ids), desc="images"))

    # Step 5: Make one record for each downloaded image
    records = []
    for i, path in fetched:
        if path is None:
            continue
        arr = np.asarray(per_image[i], dtype=np.float32)
        records.append(dict(image_path=str(path),
                            boxes=arr[:, :4].copy(),
                            labels=arr[:, 4].astype(np.int64)))
    print(f"loaded {len(records)} images, {sum(len(r['labels']) for r in records)} boxes, {len(class_names)} classes")
    return records, class_names

In [ ]:
# COCO val2017: download the annotation file and a random part of the images
COCO_DIR = CFG["ROOT"] / "coco"
ann_zip  = download_file("http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
                         COCO_DIR / "annotations_trainval2017.zip")      # about 241 MB, downloaded only once
ann_json = COCO_DIR / "annotations" / "instances_val2017.json"
if not ann_json.exists():
    with zipfile.ZipFile(ann_zip) as z:
        z.extract("annotations/instances_val2017.json", COCO_DIR)        # take out only the file we need

coco_records, coco_names = load_coco_style_subset(ann_json, COCO_DIR / "val2017", CFG["COCO_SUBSET"])
print(coco_names[:10], "...")

### 2b.LVIS minival

The paper reports its main results on **LVIS minival**. It has 1203 classes, and 337 of them are "rare", which makes it a good test for open vocabulary detection. LVIS uses the same COCO images, so the same loader works. Set `CFG["USE_LVIS"] = True` to use it.

Note: LVIS does not label every class in every image. We do not keep that extra information in our records, because only the official LVIS evaluation code needs it.

In [ ]:
lvis_records, lvis_names = [], []
if CFG["USE_LVIS"]:
    LVIS_DIR = CFG["ROOT"] / "lvis"
    # Official LVIS v1 validation file. The paper uses a 5,000 image part of it called "minival".
    # We take our own smaller random part.
    z = download_file("https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip", LVIS_DIR / "lvis_v1_val.json.zip")
    if not (LVIS_DIR / "lvis_v1_val.json").exists():
        with zipfile.ZipFile(z) as zf: zf.extractall(LVIS_DIR)
    lvis_records, lvis_names = load_coco_style_subset(LVIS_DIR / "lvis_v1_val.json",
                                                      LVIS_DIR / "images", CFG["LVIS_SUBSET"])
else:
    print("LVIS skipped (CFG['USE_LVIS'] = False)")

### 2c. Looking at the data (EDA) and data checks

Before preprocessing, we look at the full val2017 annotation file.

This section shows:
* a summary table and data checks (duplicates, missing values, images without boxes, crowd boxes, bad boxes),
* how many boxes each class has (the numbers match our EDA report),
* small, medium and large box sizes,
* plots of class counts, objects per image, image sizes and box sizes,
* a check that our 500 image subset looks like the full val2017 split.

In [ ]:
import pandas as pd

# Load the full val2017 annotation file into tables (no images needed)
with open(ann_json) as f:
    _coco = json.load(f)
images_df      = pd.DataFrame(_coco["images"])
annotations_df = pd.DataFrame(_coco["annotations"])
categories_df  = pd.DataFrame(_coco["categories"]).sort_values("id").reset_index(drop=True)
del _coco                                            # free memory, we do not need the raw JSON now

# COCO saves a box as [x, y, w, h]. We split it into 4 separate columns.
bb = pd.DataFrame(annotations_df["bbox"].tolist(), columns=["x", "y", "bw", "bh"])
ann = pd.concat([annotations_df.drop(columns="bbox"), bb], axis=1)
ann["box_area"] = ann["bw"] * ann["bh"]                                            # box area in pixels
ann = ann.merge(images_df[["id", "width", "height"]].rename(columns={"id": "image_id"}), on="image_id", how="left")
ann["rel_area"] = ann["box_area"] / (ann["width"] * ann["height"])                 # part of the image the box covers
images_df["aspect_ratio"] = images_df["width"] / images_df["height"]

MIN_SIDE = 2.0     # same limit as in load_coco_style_subset()

# Summary and data checks in one table
checks = {
    "images": len(images_df),
    "annotations": len(ann),
    "categories": len(categories_df),
    "duplicate image ids": int(images_df["id"].duplicated().sum()),
    "duplicate annotation ids": int(ann["id"].duplicated().sum()),
    "missing values in annotations": int(annotations_df.isnull().sum().sum()),
    "images without any annotation": len(set(images_df["id"]) - set(ann["image_id"])),
    "crowd annotations (removed by our loader)": int(ann["iscrowd"].sum()),
    "boxes with width or height <= 0": int(((ann["bw"] <= 0) | (ann["bh"] <= 0)).sum()),
    f"boxes with a side < {MIN_SIDE:.0f}px (removed by our loader)": int(((ann["bw"] < MIN_SIDE) | (ann["bh"] < MIN_SIDE)).sum()),
    "boxes extending outside the image": int(((ann["x"] < 0) | (ann["y"] < 0) |
                                              (ann["x"] + ann["bw"] > ann["width"]) |
                                              (ann["y"] + ann["bh"] > ann["height"])).sum()),
    "subset images missing on disk": sum(not Path(r["image_path"]).exists() for r in coco_records),
}
display(pd.Series(checks, name="value").to_frame())

In [ ]:
# How many boxes does each class have? We leave out crowd boxes and tiny boxes, like our loader does.
valid = ann[(ann["iscrowd"] == 0) & (ann["bw"] >= MIN_SIDE) & (ann["bh"] >= MIN_SIDE)]
id2name = dict(zip(categories_df["id"], categories_df["name"]))
class_counts = valid["category_id"].value_counts().rename_axis("category_id").reset_index(name="count")
class_counts["name"] = class_counts["category_id"].map(id2name)

# Our EDA report counts all 36,781 boxes. Our loader removes crowd boxes and boxes smaller than 2 px.
# We show both numbers side by side, so they can be matched with the report.
raw_counts = ann["category_id"].value_counts()
cmp = pd.DataFrame({"name": class_counts["name"],
                    "all boxes (EDA report)": class_counts["category_id"].map(raw_counts).values,
                    "kept by our loader": class_counts["count"].values})
display(cmp.head(5))
print(f"boxes kept by our loader: {len(valid):,} of {len(ann):,} ({len(valid)/len(ann):.1%})")

c = class_counts["count"]
print(f"most frequent: {class_counts.iloc[0]['name']} ({c.max()}) | least frequent: {class_counts.iloc[-1]['name']} ({c.min()}) "
      f"| mean {c.mean():.0f} | median {c.median():.0f} | imbalance max/min = {c.max()/c.min():.0f}x")

# Small, medium and large boxes. COCO limits: small is below 32x32 pixels, large is above 96x96.
small  = (valid["box_area"] <  32 ** 2).mean()
medium = ((valid["box_area"] >= 32 ** 2) & (valid["box_area"] < 96 ** 2)).mean()
large  = (valid["box_area"] >= 96 ** 2).mean()
print(f"box sizes: small {small:.1%} | medium {medium:.1%} | large {large:.1%}  "
      "(small objects become even smaller after resizing to 640 px, so AP_small matters)")
print("Note: the official COCO score uses the mask area for these groups, so its numbers are slightly different.")

# Plots
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
top = class_counts.head(20).iloc[::-1]
ax[0, 0].barh(top["name"], top["count"]); ax[0, 0].set_title("Top-20 classes (val2017)"); ax[0, 0].set_xlabel("boxes")
per_img = valid.groupby("image_id").size()
ax[0, 1].hist(per_img, bins=40); ax[0, 1].set_title("Objects per image"); ax[0, 1].set_xlabel("boxes in image")
ax[1, 0].scatter(images_df["width"], images_df["height"], alpha=0.3, s=8)
ax[1, 0].set_title("Image width vs height"); ax[1, 0].set_xlabel("width px"); ax[1, 0].set_ylabel("height px")
ax[1, 1].hist(np.log10(valid["rel_area"]), bins=50); ax[1, 1].set_title("Box area / image area (log10)")
ax[1, 1].set_xlabel("log10(relative area)  (-2 means 1% of the image)")
plt.tight_layout(); plt.show()

In [ ]:
# Is our small subset similar to the full val2017 split?
# For each class we compare its share of all boxes in the full split and in our subset.
cat_ids = list(categories_df["id"])                                  # sorted ids, same order as our class numbers 0 to 79
full_counts = valid["category_id"].value_counts().reindex(cat_ids, fill_value=0).values.astype(float)
sub_counts = np.bincount(np.concatenate([r["labels"] for r in coco_records]), minlength=len(cat_ids)).astype(float)
full_share, sub_share = full_counts / full_counts.sum(), sub_counts / sub_counts.sum()   # share of each class

r_corr = np.corrcoef(full_share, sub_share)[0, 1]    # close to 1.0 means very similar
print(f"correlation of class shares (subset vs full): {r_corr:.3f}  |  classes with 0 boxes in subset: {(sub_counts == 0).sum()}")

plt.figure(figsize=(5.5, 5))
plt.scatter(full_share, sub_share, s=14)
plt.plot([0, full_share.max()], [0, full_share.max()], "r--", lw=1)      # red line y = x means a perfect match
plt.xlabel("share in full val2017"); plt.ylabel(f"share in our {len(coco_records)}-image subset")
plt.title(f"Subset vs full split (r = {r_corr:.2f})"); plt.tight_layout(); plt.show()

# Save the tables so we can use them in the report
EDA_DIR = CFG["ROOT"] / "eda"; EDA_DIR.mkdir(exist_ok=True)
pd.Series(checks).to_csv(EDA_DIR / "dataset_checks.csv", header=["value"])
class_counts.to_csv(EDA_DIR / "class_counts_full_val2017.csv", index=False)
print("saved tables to", EDA_DIR)

In [ ]:
# Raw data examples: 4 images from our subset with their real boxes
# The boxes are drawn on the original image, before any resizing.
import matplotlib.patches as patches
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax_, rec in zip(axes, coco_records[:4]):
    img_rgb = np.asarray(Image.open(rec["image_path"]).convert("RGB"))
    ax_.imshow(img_rgb)
    for (x1, y1, x2, y2), cls in zip(rec["boxes"], rec["labels"]):
        ax_.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor="red", facecolor="none"))
        ax_.text(x1, max(y1 - 3, 0), coco_names[cls], color="white", fontsize=7,
                 bbox=dict(facecolor="red", alpha=0.6, pad=1))
    ax_.set_title(f"{Path(rec['image_path']).name} | {len(rec['labels'])} boxes", fontsize=9); ax_.axis("off")
plt.tight_layout(); plt.show()

## 3. Custom fine tuning dataset: African Wildlife

**Our choice (next step 1 of the report).** We use the *African Wildlife* dataset that Ultralytics publishes in YOLO format (about 1,500 photos; train, val and test splits). It has 4 classes: **buffalo, elephant, rhino, zebra**.

Why this dataset:
* **Two classes are in COCO and two are not.** *Elephant* and *zebra* are among COCO's 80 classes, *buffalo* and *rhino* are not. So we can compare the zero-shot and fine-tuned results for classes the COCO list knows and for classes it does not know.
* **It is small**, so fine tuning two models fits in a free GPU session.
* **It already has a test split**, so we can train on *train*, choose the best epoch on *val* and report the final numbers on *test*, which the models never saw.

Note: "not in COCO" does not mean the class is unseen by YOLO-World. Its pre-training data (Objects365, GoldG, image-text pairs) is much larger than COCO and may contain these animals. We therefore call buffalo and rhino *outside COCO*, not *unseen*.

The dataset uses the Ultralytics folder layout:
```
african-wildlife/
  images/train/*.jpg   labels/train/*.txt
  images/val/*.jpg     labels/val/*.txt
  images/test/*.jpg    labels/test/*.txt
```
Roboflow uses the other layout (`train/images`, `train/labels`, ...). Our reader below understands both. Each line in a label file is `class_id cx cy w h`, where the box centre and size are **fractions between 0 and 1**. We change them into pixel boxes `[x1, y1, x2, y2]`, the same record format as above.

To use a Roboflow Universe dataset instead, fill in the `ROBOFLOW` settings (a free API key is needed) and delete the `data/custom` folder.

In [ ]:
CUSTOM_DIR = CFG["ROOT"] / "custom"

# Our custom dataset. "names" is the class order of the label files (class 0 = buffalo, ...).
CUSTOM_DATASET = dict(
    name  = "african-wildlife",
    url   = "https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip",
    names = ["buffalo", "elephant", "rhino", "zebra"],
    # More exact wording for the prompt study in Section 10b
    prompt_overrides = {"buffalo": "african buffalo", "elephant": "african elephant", "rhino": "rhinoceros"},
)

# Optional: a Roboflow Universe dataset instead. If api_key is empty, Roboflow is not used.
ROBOFLOW = dict(api_key="", workspace="", project="", version=1)

def download_roboflow(rf_cfg, out_dir):
    """Download a Roboflow Universe dataset in YOLOv8 format into out_dir."""
    from roboflow import Roboflow            # imported here because only this step needs it
    rf = Roboflow(api_key=rf_cfg["api_key"])
    proj = rf.workspace(rf_cfg["workspace"]).project(rf_cfg["project"])
    return proj.version(rf_cfg["version"]).download("yolov8", location=str(out_dir))


def find_dataset_root(folder):
    """A zip file sometimes has an extra top folder. Return the folder that holds the splits, or None."""
    folder = Path(folder)
    if not folder.exists():
        return None
    for d in [folder, *sorted(p for p in folder.rglob("*") if p.is_dir())]:
        if (d / "images" / "train").is_dir() or (d / "train" / "images").is_dir():
            return d
    return None


if find_dataset_root(CUSTOM_DIR) is None:
    if ROBOFLOW["api_key"]:
        download_roboflow(ROBOFLOW, CUSTOM_DIR)
    else:
        zip_path = download_file(CUSTOM_DATASET["url"], CFG["ROOT"] / f"{CUSTOM_DATASET['name']}.zip")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(CUSTOM_DIR)

CUSTOM_ROOT = find_dataset_root(CUSTOM_DIR)
# Roboflow gives a data.yaml with the class names. The Ultralytics zip does not, so we write one.
if CUSTOM_ROOT is not None and not (CUSTOM_ROOT / "data.yaml").exists():
    (CUSTOM_ROOT / "data.yaml").write_text(yaml.safe_dump({"names": CUSTOM_DATASET["names"]}))
print("custom dataset folder:", CUSTOM_ROOT)

In [ ]:
SPLIT_ALIASES = {"train": ["train"], "valid": ["valid", "val"], "val": ["val", "valid"], "test": ["test"]}

def split_dirs(root, split):
    """Find the image and label folders of one split. Works with both layouts:
    Roboflow (train/images, train/labels) and Ultralytics (images/train, labels/train)."""
    root = Path(root)
    for s in SPLIT_ALIASES.get(split, [split]):
        for img_dir, lbl_dir in [(root / s / "images", root / s / "labels"),
                                 (root / "images" / s, root / "labels" / s)]:
            if img_dir.is_dir():
                return img_dir, lbl_dir
    return None, None


def load_yolo_format_split(root, split):
    """Read one split ("train", "valid" or "test") of a YOLO format dataset and return our simple records.
    Roboflow and Ultralytics give YOLO .txt label files, so we read them directly and do not need to convert them."""
    root = Path(root)
    with open(root / "data.yaml") as f:
        meta = yaml.safe_load(f)
    names = meta["names"]
    if isinstance(names, dict):                       # some datasets write names as {0: 'cat', 1: 'dog'}
        names = [names[k] for k in sorted(names)]
    class_names = [clean_class_name(n) for n in names]

    img_dir, lbl_dir = split_dirs(root, split)
    if img_dir is None:
        print(f"[{split}] split not found in {root}")
        return [], class_names
    records = []
    for img_path in sorted(img_dir.glob("*")):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            continue
        w, h = Image.open(img_path).size              # only reads the file header, so it is fast
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        rows = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:                    # skip broken lines
                    continue
                c, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                # change fractions (centre x, centre y, width, height) into pixels (left, top, right, bottom)
                rows.append([(cx - bw / 2) * w, (cy - bh / 2) * h,
                             (cx + bw / 2) * w, (cy + bh / 2) * h, c])
        arr = np.asarray(rows, dtype=np.float32).reshape(-1, 5)
        records.append(dict(image_path=str(img_path), boxes=arr[:, :4].copy(), labels=arr[:, 4].astype(np.int64)))
    print(f"[{split}] {len(records)} images, {sum(len(r['labels']) for r in records)} boxes, classes={class_names}")
    return records, class_names

custom_train, custom_val, custom_test, custom_names = [], [], [], []
if CUSTOM_ROOT is not None and (CUSTOM_ROOT / "data.yaml").exists():
    custom_train, custom_names = load_yolo_format_split(CUSTOM_ROOT, "train")
    custom_val,   _            = load_yolo_format_split(CUSTOM_ROOT, "valid")
    custom_test,  _            = load_yolo_format_split(CUSTOM_ROOT, "test")
    if CFG["QUICK_TEST"]:                             # tiny run: only a few images per split
        custom_train, custom_val, custom_test = custom_train[:24], custom_val[:8], custom_test[:8]
else:
    print("Custom dataset not found -> skipping it (the COCO parts still run).")

In [ ]:
# Test of the YOLO format reader
# This is NOT our research data. We make a tiny fake dataset (one grey picture with one box)
# in a temporary folder, to check that the reader changes fractions into the correct pixel boxes.
import tempfile, shutil

def self_test_yolo_reader():
    tmp = Path(tempfile.mkdtemp())
    for split in ["train", "valid"]:
        (tmp / split / "images").mkdir(parents=True); (tmp / split / "labels").mkdir(parents=True)
        grey = np.full((100, 200, 3), 127, np.uint8)                      # picture 200 px wide and 100 px high
        cv2.imwrite(str(tmp / split / "images" / "a.jpg"), grey)
        (tmp / split / "labels" / "a.txt").write_text("1 0.5 0.5 0.5 0.5\n")   # class 1, box in the middle, half the size
    (tmp / "data.yaml").write_text("names: ['cat', 'toy_robot']\n")
    recs, names = load_yolo_format_split(tmp, "train")
    assert names == ["cat", "toy robot"]                                    # underscore was removed
    assert np.allclose(recs[0]["boxes"][0], [50, 25, 150, 75], atol=1e-3)   # expected box in pixels
    assert recs[0]["labels"][0] == 1
    shutil.rmtree(tmp)
    print("YOLO-format reader self-test passed")

self_test_yolo_reader()

### 3b. EDA of the custom dataset (next step 1)

We repeat the EDA of Section 2c on the custom dataset:
* number of images and boxes in each split, and which classes are also in COCO's 80 classes,
* data checks: images without boxes, wrong class ids, boxes with no size, tiny boxes, boxes outside the image, and **the same image file in two splits** (this would leak test images into training),
* plots of class counts per split, objects per image, image sizes and box sizes, and some raw images with their boxes.

The tables are saved in `data/eda/` next to the COCO tables.

In [ ]:
import hashlib
import matplotlib.patches as patches

def custom_eda_tables(splits, names):
    """One table with a row per image and one table with a row per box."""
    img_rows, box_rows = [], []
    for split, recs in splits.items():
        for r in recs:
            p = Path(r["image_path"]); w, h = Image.open(p).size
            img_rows.append(dict(split=split, file=p.name, path=str(p), width=w, height=h, boxes=len(r["labels"]),
                                 md5=hashlib.md5(p.read_bytes()).hexdigest()))   # file fingerprint for the duplicate check
            for (x1, y1, x2, y2), c in zip(r["boxes"].tolist(), r["labels"].tolist()):
                box_rows.append(dict(split=split, file=p.name, cls=c,
                                     name=names[c] if 0 <= c < len(names) else "INVALID",
                                     x1=x1, y1=y1, x2=x2, y2=y2, bw=x2 - x1, bh=y2 - y1, img_w=w, img_h=h))
    imgs = pd.DataFrame(img_rows)
    boxes = pd.DataFrame(box_rows, columns=["split", "file", "cls", "name", "x1", "y1", "x2", "y2",
                                            "bw", "bh", "img_w", "img_h"])
    boxes["rel_area"] = boxes["bw"] * boxes["bh"] / (boxes["img_w"] * boxes["img_h"])
    return imgs, boxes

if custom_names:
    SPLITS = {"train": custom_train, "valid": custom_val, "test": custom_test}
    cimgs, cboxes = custom_eda_tables(SPLITS, custom_names)

    # Which custom classes are also in COCO's 80 classes?
    in_coco = {n: n in coco_names for n in custom_names}
    split_counts = (cboxes.pivot_table(index="name", columns="split", values="file", aggfunc="count", fill_value=0)
                          .reindex(index=custom_names, columns=list(SPLITS), fill_value=0))
    split_counts["total"] = split_counts.sum(axis=1)
    split_counts["in COCO 80?"] = [in_coco[n] for n in split_counts.index]
    display(split_counts)

    leak = cimgs.groupby("md5")["split"].nunique()
    custom_checks = {
        "images (train / valid / test)": " / ".join(str(int((cimgs["split"] == s).sum())) for s in SPLITS),
        "boxes": len(cboxes),
        "classes": len(custom_names),
        "classes in COCO 80": ", ".join(n for n in custom_names if in_coco[n]) or "none",
        "classes outside COCO 80": ", ".join(n for n in custom_names if not in_coco[n]) or "none",
        "images without any box": int((cimgs["boxes"] == 0).sum()),
        "labels with an invalid class id": int((cboxes["name"] == "INVALID").sum()),
        "boxes with width or height <= 0": int(((cboxes["bw"] <= 0) | (cboxes["bh"] <= 0)).sum()),
        "boxes with a side < 2 px": int(((cboxes["bw"] < 2) | (cboxes["bh"] < 2)).sum()),
        "boxes outside the image (> 1 px)": int(((cboxes["x1"] < -1) | (cboxes["y1"] < -1) |
                                                (cboxes["x2"] > cboxes["img_w"] + 1) |
                                                (cboxes["y2"] > cboxes["img_h"] + 1)).sum()),
        "duplicate image files (same content)": int(cimgs["md5"].duplicated().sum()),
        "image files found in more than one split": int((leak > 1).sum()),
    }
    display(pd.Series(custom_checks, name="value").to_frame())

    per_img = cimgs["boxes"]
    print(f"objects per image: mean {per_img.mean():.2f} | median {per_img.median():.0f} | max {per_img.max()}")
    print(f"relative box area: median {cboxes['rel_area'].median():.3f} (custom) vs "
          f"{ann['rel_area'].median():.3f} (COCO val2017) -> custom objects are "
          f"{'larger' if cboxes['rel_area'].median() > ann['rel_area'].median() else 'smaller'} on average")
    print(f"image size: median {cimgs['width'].median():.0f} x {cimgs['height'].median():.0f} px")

    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    split_counts[list(SPLITS)].plot.bar(ax=ax[0, 0], rot=0)
    ax[0, 0].set_title("Boxes per class and split"); ax[0, 0].set_ylabel("boxes")
    ax[0, 1].hist(per_img, bins=range(0, int(per_img.max()) + 2)); ax[0, 1].set_title("Objects per image")
    ax[0, 1].set_xlabel("boxes in image")
    ax[1, 0].scatter(cimgs["width"], cimgs["height"], alpha=0.3, s=8)
    ax[1, 0].set_title("Image width vs height"); ax[1, 0].set_xlabel("width px"); ax[1, 0].set_ylabel("height px")
    ax[1, 1].hist(np.log10(cboxes["rel_area"].clip(lower=1e-6)), bins=50)
    ax[1, 1].set_title("Box area / image area (log10)"); ax[1, 1].set_xlabel("log10(relative area)")
    plt.tight_layout(); plt.show()

    # Raw examples: one training image for each class, with its real boxes
    examples = [next((r for r in custom_train if c in r["labels"]), None) for c in range(len(custom_names))]
    examples = [r for r in examples if r is not None][:4]
    fig, axes = plt.subplots(1, len(examples), figsize=(4.5 * len(examples), 4.2), squeeze=False)
    for ax_, rec in zip(axes[0], examples):
        ax_.imshow(np.asarray(Image.open(rec["image_path"]).convert("RGB")))
        for (x1, y1, x2, y2), c in zip(rec["boxes"], rec["labels"]):
            ax_.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor="red", facecolor="none"))
            ax_.text(x1, max(y1 - 3, 0), custom_names[c], color="white", fontsize=8, bbox=dict(facecolor="red", alpha=0.6, pad=1))
        ax_.set_title(f"{Path(rec['image_path']).name} | {len(rec['labels'])} boxes", fontsize=9); ax_.axis("off")
    plt.tight_layout(); plt.show()

    EDA_DIR = CFG["ROOT"] / "eda"; EDA_DIR.mkdir(exist_ok=True)
    pd.Series(custom_checks).to_csv(EDA_DIR / "custom_dataset_checks.csv", header=["value"])
    split_counts.to_csv(EDA_DIR / "custom_class_counts.csv")
    print("saved custom EDA tables to", EDA_DIR)
else:
    print("No custom dataset -> custom EDA skipped.")

## 4. Image preprocessing and augmentation

YOLO-World uses the YOLOv8 image steps. For **training** we do these steps in order:

1. **Mosaic.** Put 4 random images together on one big canvas (2S x 2S, where S = 640). The model sees many objects and backgrounds in one picture.
2. **Random zoom and shift.** Zoom between 0.5x and 1.5x and move the picture a little. This also cuts the big canvas back to S x S.
3. **Colour change (HSV).** Randomly change hue, saturation and brightness.
4. **Horizontal flip** (half of the time).
5. **Scale the pixels.** Convert to RGB and divide by 255, so every value is between 0 and 1. YOLO does not subtract a mean or divide by a standard deviation.

For **evaluation** we only use fixed steps: **letterbox** (resize the image to fit inside S x S, keep its shape, and fill the empty space with grey) and divide by 255. We keep the scale and padding, so predicted boxes can be moved back to the original image.

**Steps we did not add:** the official config also has some rare effects (blur, median blur, grey and CLAHE, each used only 1% of the time), CopyPaste (needs object masks) and MixUp (15% of the time). They change the result very little and need extra libraries. In the last epochs the official training turns mosaic off; our `mosaic_prob` setting can do the same.

The boxes are changed together with the image in every step. After a step that moves or cuts the image, boxes are cut at the image border. A box is removed if it becomes too small or if most of it is cut off.

In [ ]:
def letterbox(img, new_size=640, pad_value=114):
    """Resize the image to fit inside new_size x new_size without changing its shape,
    then fill the empty space with grey.
    Returns the new image, the scale we used, and the padding (left, top)."""
    h, w = img.shape[:2]
    r = min(new_size / h, new_size / w)                       # one scale for width and height, so the shape stays the same
    nw, nh = int(round(w * r)), int(round(h * r))
    if (nw, nh) != (w, h):
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    dw, dh = (new_size - nw) / 2, (new_size - nh) / 2          # split the padding equally on both sides
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))   # small trick so the total is exactly new_size
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value,) * 3)
    return img, r, (left, top)


def augment_hsv(img, hgain=0.015, sgain=0.7, vgain=0.4):
    """Randomly change the colour in HSV (hue, saturation, value). The gains are the YOLOv8 defaults.
    Input and output are RGB images with values from 0 to 255."""
    r = np.random.uniform(-1, 1, 3) * np.array([hgain, sgain, vgain]) + 1     # three random numbers close to 1.0
    hue, sat, val = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2HSV))
    x = np.arange(0, 256, dtype=r.dtype)
    # A lookup table tells the new value for every old value. It is much faster than changing each pixel in a loop.
    lut_h = ((x * r[0]) % 180).astype(np.uint8)               # hue goes round in a circle (0 to 179 in OpenCV)
    lut_s = np.clip(x * r[1], 0, 255).astype(np.uint8)
    lut_v = np.clip(x * r[2], 0, 255).astype(np.uint8)
    hsv = cv2.merge((cv2.LUT(hue, lut_h), cv2.LUT(sat, lut_s), cv2.LUT(val, lut_v)))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def random_affine(img, boxes, labels, out_size=640, scale=0.5, translate=0.1,
                  min_wh=2.0, min_area_ratio=0.1, max_aspect=100.0):
    """Random zoom and shift. There is no rotation, so boxes stay straight rectangles.
    The centre of the image is moved to a random point near the middle of the out_size x out_size picture.
    This also cuts the big mosaic canvas back to S x S.
    min_wh, min_area_ratio and max_aspect use the same values as the official YOLO-World config."""
    h, w = img.shape[:2]
    s  = random.uniform(1 - scale, 1 + scale)                          # zoom factor
    tx = random.uniform(0.5 - translate, 0.5 + translate) * out_size   # new x position of the image centre
    ty = random.uniform(0.5 - translate, 0.5 + translate) * out_size   # new y position of the image centre
    ox, oy = tx - s * w / 2, ty - s * h / 2                            # new top left corner after zoom
    M = np.array([[s, 0, ox], [0, s, oy]], dtype=np.float32)           # matrix that does zoom and then shift
    out = cv2.warpAffine(img, M, (out_size, out_size), flags=cv2.INTER_LINEAR,
                         borderValue=(CFG["PAD_VALUE"],) * 3)
    if len(boxes) == 0:
        return out, boxes, labels
    nb = boxes.copy()
    nb[:, [0, 2]] = nb[:, [0, 2]] * s + ox                              # move the boxes in the same way
    nb[:, [1, 3]] = nb[:, [1, 3]] * s + oy
    area_before = (nb[:, 2] - nb[:, 0]) * (nb[:, 3] - nb[:, 1])         # box area before cutting at the border
    nb[:, [0, 2]] = nb[:, [0, 2]].clip(0, out_size)                     # cut the boxes at the picture border
    nb[:, [1, 3]] = nb[:, [1, 3]].clip(0, out_size)
    bw, bh = nb[:, 2] - nb[:, 0], nb[:, 3] - nb[:, 1]
    # Keep a box only if it is big enough, at least 10% of it is still visible, and it is not a very thin line
    keep = (bw > min_wh) & (bh > min_wh) & (bw * bh / (area_before + 1e-9) > min_area_ratio) \
           & (np.maximum(bw / (bh + 1e-9), bh / (bw + 1e-9)) < max_aspect)
    return out, nb[keep], labels[keep]


def horizontal_flip(img, boxes, p=0.5):
    """Mirror the image from left to right with probability p. An x position becomes (width minus x)."""
    if random.random() < p:
        img = img[:, ::-1]
        if len(boxes):
            w = img.shape[1]
            boxes = boxes.copy()
            boxes[:, [0, 2]] = w - boxes[:, [2, 0]]     # swap left and right so x1 stays smaller than x2
    return np.ascontiguousarray(img), boxes


def mosaic4(samples, s=640):
    """Put 4 images (with their boxes and labels) on one 2s x 2s canvas around a random centre point (xc, yc).
    Each image is already resized so its longest side is s.
    After this, random_affine cuts the canvas back to s x s.
    This follows the mosaic method of YOLOv5 and YOLOv8."""
    canvas = np.full((2 * s, 2 * s, 3), CFG["PAD_VALUE"], dtype=np.uint8)
    xc, yc = [int(random.uniform(0.5 * s, 1.5 * s)) for _ in range(2)]    # random centre of the mosaic
    all_boxes, all_labels = [], []
    for i, (img, boxes, labels) in enumerate(samples):
        h, w = img.shape[:2]
        # "a" is the place on the canvas where we paste. "b" is the part of the image that we copy.
        if i == 0:   # top left
            x1a, y1a, x2a, y2a = max(xc - w, 0), max(yc - h, 0), xc, yc
            x1b, y1b, x2b, y2b = w - (x2a - x1a), h - (y2a - y1a), w, h
        elif i == 1: # top right
            x1a, y1a, x2a, y2a = xc, max(yc - h, 0), min(xc + w, 2 * s), yc
            x1b, y1b, x2b, y2b = 0, h - (y2a - y1a), min(w, x2a - x1a), h
        elif i == 2: # bottom left
            x1a, y1a, x2a, y2a = max(xc - w, 0), yc, xc, min(2 * s, yc + h)
            x1b, y1b, x2b, y2b = w - (x2a - x1a), 0, w, min(y2a - y1a, h)
        else:        # bottom right
            x1a, y1a, x2a, y2a = xc, yc, min(xc + w, 2 * s), min(2 * s, yc + h)
            x1b, y1b, x2b, y2b = 0, 0, min(w, x2a - x1a), min(y2a - y1a, h)
        canvas[y1a:y2a, x1a:x2a] = img[y1b:y2b, x1b:x2b]
        padw, padh = x1a - x1b, y1a - y1b                  # how far the image moved when we pasted it
        if len(boxes):
            b = boxes.copy()
            b[:, [0, 2]] += padw; b[:, [1, 3]] += padh      # move the boxes by the same amount
            all_boxes.append(b); all_labels.append(labels)
    if all_boxes:
        boxes, labels = np.concatenate(all_boxes), np.concatenate(all_labels)
        # Cut boxes at the canvas border. Small or hidden boxes are removed later in random_affine.
        boxes = boxes.clip(0, 2 * s)
    else:
        boxes, labels = np.zeros((0, 4), np.float32), np.zeros((0,), np.int64)
    return canvas, boxes, labels

### 4b. YOLO-World text sampling

This step is special to YOLO-World. In the official training, every image gets its **own list of class names**:

1. take the classes that are really in the image (**positives**),
2. add random classes that are **not** in the image (**negatives**), until there are 80 names,
3. shuffle the list, and **change each box label to the position of its class in this list**,
4. if the list is shorter than 80, fill it with empty names `""` that the model ignores.

The negatives teach the model that a region is *not* some other class. This helps it work with any list of class names. For **evaluation**, every image simply uses the full class list in the normal order, so labels do not change.

Our function `random_load_text` does the same thing as `RandomLoadText` in the official file `yolo_world/datasets/transformers/mm_transforms.py`.

In [ ]:
def random_load_text(labels, class_names, max_num_samples=80, num_neg_samples=None, pad_value=""):
    """Make the class name list for one training image (positives and negatives) and renumber its labels.
    labels: class numbers of the boxes that are still in the image after augmentation.
    Returns new_labels (positions in texts), texts (the name list) and keep (which boxes stay)."""
    n_cls = len(class_names)
    if num_neg_samples is None:
        num_neg_samples = (n_cls, n_cls)              # same setting as the official config
    positives = set(int(l) for l in labels)
    if len(positives) > max_num_samples:              # if the image has too many classes, keep a random part
        positives = set(random.sample(sorted(positives), k=max_num_samples))
    n_neg = min(min(n_cls, max_num_samples) - len(positives), random.randint(*num_neg_samples))
    negatives = random.sample([c for c in range(n_cls) if c not in positives], k=n_neg)
    sampled = list(positives) + negatives
    random.shuffle(sampled)                           # shuffle so the real classes are not always first
    label2pos = {c: i for i, c in enumerate(sampled)}
    keep = np.array([int(l) in label2pos for l in labels], dtype=bool)
    new_labels = np.array([label2pos[int(l)] for l in labels[keep]], dtype=np.int64)
    texts = [class_names[c] for c in sampled]
    texts += [pad_value] * (max_num_samples - len(texts))   # fill up to 80 names with empty text
    return new_labels, texts, keep


# Test: after renumbering, every box must still point to the name of its real class
_lab = np.array([0, 0, 15, 16])                       # person, person, cat, dog (COCO numbers)
_names = [f"class{i}" for i in range(80)]
_new, _texts, _keep = random_load_text(_lab, _names)
assert _keep.all() and len(_texts) == 80
assert [_texts[i] for i in _new] == [_names[i] for i in _lab]
_new, _texts, _keep = random_load_text(np.array([1, 2]), ["cat", "dog", "toy robot"])   # small custom class list
assert len(_texts) == 80 and _texts.count("") == 77 and [_texts[i] for i in _new] == ["dog", "toy robot"]
print("random_load_text self-test passed")

## 5. PyTorch `Dataset` and `DataLoader`

`DetectionDataset` takes our records and gives back, for each image:

* `img`: a `float32` tensor with shape `(3, S, S)`, RGB, values between 0 and 1
* `targets`: a tensor `(n, 5)` with `[class, cx, cy, w, h]`. The box numbers are **fractions of the image size**, which is the YOLO label style
* `texts`: the class names for this image. In training this is the list from `random_load_text`, and the class numbers in `targets` point into it. In evaluation it is the full class list
* `meta`: the file path, the original size, and the letterbox scale and padding. We need these to move predictions back to the original image

`augment=False` uses only letterbox (for evaluation). `augment=True` uses the training steps from Section 4.

`collate_fn` joins the samples into one batch. All images have the same size, so they can be stacked. But every image has a different number of boxes, so we put all boxes in one long table, with a first column that says which image the box belongs to. The name lists are kept as a list of lists.

In [ ]:
class DetectionDataset(Dataset):
    def __init__(self, records, class_names, img_size=640, augment=False, mosaic_prob=1.0, flip_prob=0.5,
                 text_sampling=True, max_texts=80):
        self.records, self.class_names = records, class_names
        self.s, self.augment = img_size, augment
        self.mosaic_prob, self.flip_prob = mosaic_prob, flip_prob
        self.text_sampling, self.max_texts = text_sampling, max_texts   # YOLO-World text step (Section 4b)

    def __len__(self):
        return len(self.records)

    # Small helper functions
    def _read_rgb(self, idx):
        img = cv2.imread(self.records[idx]["image_path"])           # OpenCV reads colours as BGR
        if img is None:
            raise FileNotFoundError(self.records[idx]["image_path"])
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)                  # the model needs RGB

    def _load_resized(self, idx):
        """Read an image and resize it so its longest side is img_size (shape kept, no padding)."""
        img = self._read_rgb(idx)
        h, w = img.shape[:2]
        r = self.s / max(h, w)
        if r != 1:
            img = cv2.resize(img, (int(round(w * r)), int(round(h * r))), interpolation=cv2.INTER_LINEAR)
        rec = self.records[idx]
        return img, rec["boxes"] * r, rec["labels"].copy()

    # Main function: PyTorch calls this to get one sample
    def __getitem__(self, idx):
        rec = self.records[idx]
        if not self.augment:
            # Evaluation: only letterbox, nothing random
            img = self._read_rgb(idx)
            h0, w0 = img.shape[:2]
            img, r, (padx, pady) = letterbox(img, self.s, CFG["PAD_VALUE"])
            boxes = rec["boxes"] * r                                  # boxes get the same scale as the image
            boxes[:, [0, 2]] += padx; boxes[:, [1, 3]] += pady        # and the same padding shift
            labels = rec["labels"]
            meta = dict(path=rec["image_path"], orig_size=(h0, w0), ratio=r, pad=(padx, pady))
        else:
            # Training
            if random.random() < self.mosaic_prob:
                # Mosaic: this image and 3 random others, then zoom and cut back to S x S
                ids = [idx] + [random.randrange(len(self)) for _ in range(3)]
                canvas, boxes, labels = mosaic4([self._load_resized(i) for i in ids], self.s)
                img, boxes, labels = random_affine(canvas, boxes, labels, self.s)
            else:
                # One image only: letterbox to S x S, then random zoom and shift
                img, boxes, labels = self._load_resized(idx)
                h, w = img.shape[:2]
                img, _, (padx, pady) = letterbox(img, self.s, CFG["PAD_VALUE"])
                if len(boxes):
                    boxes = boxes.copy(); boxes[:, [0, 2]] += padx; boxes[:, [1, 3]] += pady
                img, boxes, labels = random_affine(img, boxes, labels, self.s)
            img = augment_hsv(img)                                    # colour change
            img, boxes = horizontal_flip(img, boxes, self.flip_prob)  # random mirror
            meta = dict(path=rec["image_path"], orig_size=None, ratio=1.0, pad=(0, 0))

        # YOLO-World class names: random positives and negatives for training, the full list for evaluation
        if self.augment and self.text_sampling:
            labels, texts, keep = random_load_text(np.asarray(labels), self.class_names, self.max_texts)
            boxes = np.asarray(boxes)[keep]
        else:
            texts = list(self.class_names)

        # For both: scale the pixels to 0 to 1 and change boxes into fractions (cx, cy, w, h)
        img = np.ascontiguousarray(img.transpose(2, 0, 1))            # change (height, width, colour) to (colour, height, width)
        img = torch.from_numpy(img).float() / 255.0                   # pixel values 0 to 255 become 0 to 1
        n = len(boxes)
        tgt = torch.zeros((n, 5), dtype=torch.float32)
        if n:
            b = np.asarray(boxes, dtype=np.float32) / self.s          # divide by the image size to get fractions
            tgt[:, 0] = torch.from_numpy(np.asarray(labels, dtype=np.float32))
            tgt[:, 1] = torch.from_numpy((b[:, 0] + b[:, 2]) / 2)     # centre x
            tgt[:, 2] = torch.from_numpy((b[:, 1] + b[:, 3]) / 2)     # centre y
            tgt[:, 3] = torch.from_numpy(b[:, 2] - b[:, 0])           # width
            tgt[:, 4] = torch.from_numpy(b[:, 3] - b[:, 1])           # height
        return dict(img=img, targets=tgt, texts=texts, meta=meta)


def collate_fn(batch):
    """Stack the images into one tensor. Put all boxes in one table whose first column is the image number."""
    imgs = torch.stack([b["img"] for b in batch])
    tg = []
    for i, b in enumerate(batch):
        t = b["targets"]
        tg.append(torch.cat([torch.full((len(t), 1), float(i)), t], dim=1))
    return dict(img=imgs, targets=torch.cat(tg) if tg else torch.zeros((0, 6)),
                texts=[b["texts"] for b in batch], meta=[b["meta"] for b in batch])


def seed_worker(worker_id):
    """Give every loader worker its own random seed, so the workers do not repeat the same random numbers."""
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


def make_loader(records, names, train):
    ds = DetectionDataset(records, names, CFG["IMG_SIZE"], augment=train)
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ds, batch_size=CFG["BATCH_SIZE"], shuffle=train, num_workers=CFG["NUM_WORKERS"],
                      collate_fn=collate_fn, worker_init_fn=seed_worker, generator=g, drop_last=False)

In [ ]:
# Loader for testing: no augmentation, on the COCO subset
coco_eval_loader = make_loader(coco_records, coco_names, train=False)
# The same data with the training steps. We use it only to see what the augmentation does.
coco_aug_loader  = make_loader(coco_records, coco_names, train=True)

# Loaders for fine tuning (only made when the custom dataset is added)
custom_train_loader = make_loader(custom_train, custom_names, train=True)  if custom_train else None
custom_val_loader   = make_loader(custom_val,   custom_names, train=False) if custom_val   else None

print("eval batches:", len(coco_eval_loader), "| aug batches:", len(coco_aug_loader))

# Show the class name list that one training image gets
_s = coco_aug_loader.dataset[0]
_cls = _s["targets"][:, 0].long().tolist()
print("training sample:", len(_s["texts"]), "texts, first 5:", _s["texts"][:5])
print("its boxes now point to:", sorted({_s["texts"][c] for c in _cls}))

## 6. Text preprocessing: prompts, CLIP tokens and saved embeddings

In YOLO-World, the score of a class is how similar an **image region embedding** is to a **text embedding** of the class name. With "prompt then detect", the list of class names is turned into text embeddings **once, before detection**, so the text model does not run for every image. Our steps:

1. **Prompt template.** Put the class name in a sentence, for example `"{}"` or `"a photo of a {}"`. We will change this in our prompt experiment later.
2. **Tokenize** with the CLIP tokenizer. It cuts the text into small pieces called tokens. The maximum is 77 tokens, and shorter texts are padded.
3. **Encode** with the CLIP text model. Each prompt becomes a vector of 512 numbers. We **normalise** each vector to length 1.
4. **Save** the result to disk, so we do not compute it again later.

In [ ]:
PROMPT_TEMPLATES = {
    "plain":  "{}",                    # only the class name (YOLO-World uses this for COCO and LVIS)
    "photo":  "a photo of a {}",       # a sentence in the CLIP style
    "object": "a {} object",
    # Phrases like "brown dog" are different for each class, so they are given
    # as a dictionary {class_name: phrase} in the overrides argument below.
}

def build_prompts(class_names, template="plain", overrides=None):
    """Turn class names into prompts. overrides can replace some classes with our own phrase,
    for example {"dog": "brown dog"}."""
    t = PROMPT_TEMPLATES.get(template, template)       # we can also pass our own template text
    overrides = overrides or {}
    return [overrides.get(n, t.format(n)) for n in class_names]


_clip = {}   # we load the CLIP model only the first time and keep it here

def _load_clip(name="openai/clip-vit-base-patch32"):
    if not _clip:
        from transformers import CLIPTokenizer, CLIPTextModelWithProjection
        _clip["tok"] = CLIPTokenizer.from_pretrained(name)
        _clip["model"] = CLIPTextModelWithProjection.from_pretrained(name).to(DEVICE).eval()
    return _clip["tok"], _clip["model"]


@torch.no_grad()      # we only use the model, we do not train it, so no gradients are needed
def encode_texts(prompts, batch=256):
    """Turn prompts into normalised CLIP text embeddings with shape (number of prompts, 512)."""
    tok, model = _load_clip()
    out = []
    for i in range(0, len(prompts), batch):              # small groups, so even 1203 LVIS classes fit in memory
        t = tok(prompts[i:i + batch], padding="max_length", max_length=77,
                truncation=True, return_tensors="pt").to(DEVICE)
        e = model(**t).text_embeds                       # one vector for each prompt
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu())   # length 1, so a dot product gives cosine similarity
    return torch.cat(out)


def cached_text_embeddings(tag, class_names, template="plain", overrides=None):
    """Load the embeddings from disk if we saved them before. If not, compute them and save them.
    When overrides are used, we do not use the saved file because the prompts are different."""
    f = CFG["ROOT"] / "text_cache" / f"{tag}_{template}.pt"
    f.parent.mkdir(exist_ok=True)
    if f.exists() and not overrides:
        return torch.load(f)
    emb = encode_texts(build_prompts(class_names, template, overrides))
    if not overrides:
        torch.save(emb, f)
    return emb

In [ ]:
# Show what the tokenizer does with a few COCO class names, then make embeddings for all 80 COCO classes
tok, _ = _load_clip()
demo = build_prompts(coco_names[:3], "photo")
enc = tok(demo, padding="max_length", max_length=77, truncation=True)
for k, (p, ids) in enumerate(zip(demo, enc["input_ids"])):
    n_real = sum(enc["attention_mask"][k])
    print(f"{p!r:28} -> {n_real} real tokens (the rest is padding), first ids: {ids[:n_real]}")

coco_text_emb = cached_text_embeddings("coco", coco_names, "plain")
print("COCO text embeddings:", tuple(coco_text_emb.shape), "| lengths (should be about 1):", coco_text_emb.norm(dim=-1)[:3])

# Simple check: "cat" should be closer to "dog" than to "traffic light"
sim = coco_text_emb @ coco_text_emb.T
a, b, c = coco_names.index("cat"), coco_names.index("dog"), coco_names.index("traffic light")
print(f"similarity(cat, dog) = {sim[a,b]:.3f}   vs   similarity(cat, traffic light) = {sim[a,c]:.3f}")

### 6b. Check with the official class names

The official YOLO-World code keeps the COCO class names in the file `data/texts/coco_class_texts.json`. We download this file and check that our class names are **exactly** the same and in the same order. If they were different, our text embeddings would not match the ones the model was tested with.

In [ ]:
OFFICIAL_TEXTS_URL = "https://raw.githubusercontent.com/AILab-CVC/YOLO-World/master/data/texts/coco_class_texts.json"
try:
    official = [t[0] for t in requests.get(OFFICIAL_TEXTS_URL, timeout=30).json()]   # the file looks like [["person"], ["bicycle"], ...]
    diff = [(i, a, b) for i, (a, b) in enumerate(zip(coco_names, official)) if a != b]
    assert len(official) == len(coco_names) and not diff, f"different class texts: {diff}"
    print(f"our {len(coco_names)} COCO class names match the official coco_class_texts.json exactly")
except requests.RequestException as e:
    print("could not download the official class list (no internet?):", repr(e))

## 7. Checks

### 7a. Look at the pictures
Top row: the **evaluation** steps (letterbox only). Bottom row: the **training** steps (mosaic, zoom and shift, colour change and flip). After every step, the boxes must still be on the objects. If they are not, there is a mistake in the box code.

In [ ]:
def show_batch(batch, class_names, n=4, title=""):
    """Draw the boxes on the first n images of a batch.
    The class name is taken from batch["texts"], because in training each image has its own name list."""
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4.2))
    for i in range(n):
        im = (batch["img"][i].permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()   # back to (height, width, colour) and 0 to 255
        S = im.shape[0]
        t = batch["targets"]; t = t[t[:, 0] == i]                  # only the boxes of image number i
        for _, c, cx, cy, w, h in t.tolist():
            x1, y1, x2, y2 = [int(v * S) for v in (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)]   # fractions to pixels
            cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 2)
            name = batch["texts"][i][int(c)]                            # class number to the name for this image
            cv2.putText(im, name, (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        axes[i].imshow(im); axes[i].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_batch(next(iter(coco_eval_loader)), coco_names, title="Evaluation steps (letterbox only)")
show_batch(next(iter(coco_aug_loader)),  coco_names, title="Training steps (mosaic + zoom/shift + colour + flip)")

### 7b. Automatic checks and loading speed
The `assert` lines stop the notebook with an error if something is wrong. They find common data mistakes early: wrong shape, wrong pixel range, boxes outside the image, and class numbers that do not exist.

In [ ]:
def check_loader(loader, n_classes, name, n_batches=3):
    """Check a few batches and print how fast the data loads."""
    t0, n_img = time.time(), 0
    for bi, b in enumerate(loader):
        x, t = b["img"], b["targets"]
        S = CFG["IMG_SIZE"]
        assert x.shape[1:] == (3, S, S),            f"bad image shape {tuple(x.shape)}"
        assert x.dtype == torch.float32,             "images must be float32"
        assert 0.0 <= x.min() and x.max() <= 1.0,    "pixel values must be between 0 and 1"
        if len(t):
            assert t.shape[1] == 6,                  "targets must be (N, 6): image number, class, cx, cy, w, h"
            assert (t[:, 1] >= 0).all() and (t[:, 1] < n_classes).all(), "class number out of range"
            assert (t[:, 2:] >= -1e-4).all() and (t[:, 2:] <= 1 + 1e-4).all(), "box fractions outside 0-1"
            assert (t[:, 4] > 0).all() and (t[:, 5] > 0).all(), "a box has zero size"
            for row in t:                                                # every label must point to a real name, not an empty one
                assert b["texts"][int(row[0])][int(row[1])] != "", "a box points to a padding text"
        n_img += len(x)
        if bi + 1 >= n_batches: break
    dt = time.time() - t0
    print(f"[{name}] OK | {n_img} images in {dt:.1f}s ({n_img/dt:.1f} img/s) | last batch: "
          f"img {tuple(x.shape)}, targets {tuple(t.shape)}, pixel range [{x.min():.2f}, {x.max():.2f}]")

check_loader(coco_eval_loader, len(coco_names), "COCO eval")
check_loader(coco_aug_loader,  len(coco_names), "COCO train-aug")
if custom_train_loader: check_loader(custom_train_loader, len(custom_names), "custom train")
if custom_val_loader:   check_loader(custom_val_loader,   len(custom_names), "custom val")

# Numbers about our subset for the report
counts = np.bincount(np.concatenate([r["labels"] for r in coco_records]), minlength=len(coco_names))
print("\nCOCO subset: top-5 classes:", [(coco_names[i], int(counts[i])) for i in counts.argsort()[::-1][:5]])
print("COCO subset: classes with 0 boxes:", int((counts == 0).sum()), "of", len(coco_names))

### 7c. Test run with the pretrained YOLO-World
We are not training yet, but our setup must be able to run the model on our data. We use the released weights through Ultralytics (`yolov8s-worldv2.pt`; `m` and `l` sizes also exist). `set_classes()` turns our class names into text embeddings once, before detection. This is the "prompt then detect" step from Section 6.

We also compare the model's text embeddings with the ones we made in Section 6. Both use CLIP ViT-B/32, so they should be the same (cosine similarity close to 1).

The first time, Ultralytics downloads the weights and its CLIP package, so this cell needs internet. If it fails, it prints a message and the rest of the notebook still works.

In [ ]:
try:
    from ultralytics import YOLOWorld

    model = YOLOWorld("yolov8s-worldv2.pt")      # weights are downloaded the first time
    model.set_classes(coco_names)                 # set the class list once before detection

    # Are the model's text embeddings the same as ours from Section 6?
    yw_txt = model.model.txt_feats.reshape(len(coco_names), -1).float().cpu()
    yw_txt = torch.nn.functional.normalize(yw_txt, dim=-1)
    cos = (yw_txt * coco_text_emb).sum(-1)
    print(f"cosine(our CLIP embeddings, YOLO-World's) : min {cos.min():.4f} | mean {cos.mean():.4f}")

    # Run the model on 3 images from our subset (the same files our Dataset reads)
    sample_paths = [r["image_path"] for r in coco_records[:3]]
    results = model.predict(sample_paths, conf=0.25, imgsz=CFG["IMG_SIZE"], device=DEVICE, verbose=False)

    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5))
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, results):
        ax.imshow(r.plot()[:, :, ::-1])           # r.plot() gives BGR colours, [::-1] changes them to RGB
        ax.set_title(f"{len(r.boxes)} detections"); ax.axis("off")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("The YOLO-World test run did not work:", repr(e))
    print("Check that the install cell was run and that the computer is online, then run this cell again.")

### 7d. Save the data in YOLO format
The Ultralytics library reads datasets in YOLO format, so we save our records once in that format. Images are linked instead of copied to save disk space. If linking is not allowed, they are copied.

In [ ]:
def export_yolo_format(records, class_names, out_dir, split="val"):
    """Save the records as out/images/split/*.jpg, out/labels/split/*.txt and out/data.yaml."""
    out = Path(out_dir).resolve()
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    for r in records:
        src = Path(r["image_path"]).resolve()
        dst = out / "images" / split / src.name
        if not dst.exists():
            try: dst.symlink_to(src)                                 # a link uses no extra disk space
            except OSError:
                import shutil; shutil.copy(src, dst)                 # copy if links are not allowed (for example on Windows)
        w, h = Image.open(src).size
        b = r["boxes"]
        lines = [f"{int(c)} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"
                 for (x1, y1, x2, y2), c in zip(b, r["labels"])]      # back to fractions: centre x, centre y, width, height
        (out / "labels" / split / (src.stem + ".txt")).write_text("\n".join(lines))
    cfg = {"path": str(out), "train": f"images/{split}", "val": f"images/{split}",
           "names": {i: n for i, n in enumerate(class_names)}}
    with open(out / "data.yaml", "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out / "data.yaml"

coco_yaml = export_yolo_format(coco_records, coco_names, CFG["ROOT"] / "coco_subset_yolo", split="val")
print("wrote", coco_yaml)

# Quick check that the saved labels are correct: run the pretrained model on our subset and get its mAP.
# If the boxes were saved wrongly, the mAP would be close to 0. This only checks our pipeline.
# The full evaluation in the next part will use the official COCO tools on the original annotation file.
RUN_QUICK_VAL = True          # takes a few minutes on CPU, less than a minute on GPU
if RUN_QUICK_VAL:
    try:
        m = model.val(data=str(coco_yaml), imgsz=CFG["IMG_SIZE"], batch=CFG["BATCH_SIZE"], device=DEVICE,
                      workers=CFG["NUM_WORKERS"], plots=False, verbose=False)
        print(f"zero-shot on our {len(coco_records)}-image subset: mAP50-95 = {m.box.map:.3f} | mAP50 = {m.box.map50:.3f}")
    except Exception as e:
        print("quick validation did not run:", repr(e))

## 8. Experiment setup

From here on we run the experiments planned in the report. This section only defines shared code:

* **Model loading.** `load_world(size, prompts)` loads the released `yolov8{s,m,l}-worldv2.pt` weights and sets the vocabulary once (*prompt then detect*).
* **Official COCO evaluation.** The model runs on the original image files of our 500-image subset. Ultralytics letterboxes each image to 640 px and maps the boxes back to the original pixels. We write the boxes in the COCO result format and score them with **pycocotools** against the **original** annotation file, so crowd boxes and the small/medium/large groups are handled the official way. Only our subset images are scored (`params.imgIds`).
* **Official LVIS evaluation** with the `lvis` package, which also gives AP for rare, common and frequent classes.
* **Custom data evaluation** with the Ultralytics validator on the custom *test* split (mAP@[.5:.95], mAP50, AP per class).
* **Speed.** Batch size 1, 640 px, after 10 warm-up images. The images are read into memory first, so disk speed does not count.

Settings for every run are stored in `RUN_INFO` and saved with the results, as promised in the report (hardware, model, input size, score limit).

In [ ]:
import gc, io, contextlib, platform
import ultralytics
from ultralytics import YOLO, YOLOWorld
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

ULTRA_DEVICE = 0 if DEVICE == "cuda" else "cpu"          # Ultralytics wants a GPU number or "cpu"
RESULTS_DIR = (CFG["ROOT"] / "results").resolve(); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR    = (CFG["ROOT"] / "runs").resolve()           # Ultralytics training and validation runs
RESULTS = {}                                             # every result table goes here and is saved in Section 13

RUN_INFO = dict(
    date=time.strftime("%Y-%m-%d %H:%M"), python=platform.python_version(),
    torch=torch.__version__, ultralytics=ultralytics.__version__,
    hardware=torch.cuda.get_device_name(0) if DEVICE == "cuda" else (platform.processor() or "CPU"),
    img_size=CFG["IMG_SIZE"], conf_eval=CFG["CONF_EVAL"], conf_speed=CFG["CONF_SPEED"], max_det=CFG["MAX_DET"],
    coco_images=len(coco_records), lvis_images=len(lvis_records),
    custom_dataset=CUSTOM_DATASET["name"] if custom_names else None, epochs=CFG["EPOCHS"],
    quick_test=CFG["QUICK_TEST"],
)
print(json.dumps(RUN_INFO, indent=1))
if CFG["QUICK_TEST"]:
    print("\nQUICK_TEST is on: the numbers below only show that the code runs. They are NOT results.")


def quiet():
    """Hide the long text that pycocotools and lvis print."""
    return contextlib.redirect_stdout(io.StringIO())


def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def load_world(size="s", prompts=None):
    """Load a pretrained YOLO-World v2 model and set its vocabulary once (prompt then detect)."""
    model = YOLOWorld(f"yolov8{size}-worldv2.pt")
    if prompts is not None:
        model.set_classes(list(prompts))
    return model


def count_params(model):
    """Number of model parameters in millions."""
    return sum(p.numel() for p in model.model.parameters()) / 1e6


def predict_records(model, records, desc="predict"):
    """Run the model on the original image files of our records.
    Returns (path, boxes xyxy in original pixels, scores, class numbers) for every image."""
    paths, out, step = [r["image_path"] for r in records], [], 16
    for i in tqdm(range(0, len(paths), step), desc=desc, leave=False):
        res = model.predict(paths[i:i + step], conf=CFG["CONF_EVAL"], iou=0.7, max_det=CFG["MAX_DET"],
                            imgsz=CFG["IMG_SIZE"], device=ULTRA_DEVICE, verbose=False)
        for p, r in zip(paths[i:i + step], res):
            b = r.boxes
            out.append((p, b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)))
    return out


def to_coco_detections(preds, file_to_id, idx_to_cat):
    """Change predictions into the COCO result format: [x, y, w, h] in pixels and the real category id."""
    dets = []
    for path, xyxy, scores, classes in preds:
        img_id = file_to_id[Path(path).name]
        for (x1, y1, x2, y2), s, c in zip(xyxy.tolist(), scores.tolist(), classes.tolist()):
            dets.append(dict(image_id=img_id, category_id=idx_to_cat[c],
                             bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
                             score=round(s, 5)))
    return dets


# ---------- COCO (official pycocotools evaluation) ----------
_COCO = {}
def coco_gt():
    """Load the original val2017 annotation file once."""
    if not _COCO:
        with quiet():
            gt = COCO(str(ann_json))
        _COCO.update(gt=gt,
                     file_to_id={Path(im["coco_url"]).name: im["id"] for im in gt.dataset["images"]},
                     idx_to_cat=sorted(gt.getCatIds()))    # our class number -> COCO id (same order as our loader)
    return _COCO

COCO_KEYS = ["AP", "AP50", "AP75", "APs", "APm", "APl", "AR100"]

def coco_evaluate(dets, img_ids):
    """Official COCO box evaluation on our image subset. Returns the summary (in %) and AP per class (in %)."""
    G = coco_gt()
    n_cls = len(G["idx_to_cat"])
    if not dets:
        return dict.fromkeys(COCO_KEYS, 0.0), np.zeros(n_cls)
    with quiet():
        E = COCOeval(G["gt"], G["gt"].loadRes(dets), "bbox")
        E.params.imgIds = sorted(img_ids)
        E.evaluate(); E.accumulate(); E.summarize()
    s = np.where(E.stats < 0, np.nan, E.stats * 100)       # -1 means "no objects of this kind"
    summary = dict(zip(COCO_KEYS, [s[0], s[1], s[2], s[3], s[4], s[5], s[8]]))
    prec = E.eval["precision"][:, :, :, 0, -1]             # IoU x recall x class, all sizes, 100 detections
    per_class = np.array([prec[:, :, k][prec[:, :, k] > -1].mean() * 100 if (prec[:, :, k] > -1).any() else np.nan
                          for k in range(prec.shape[2])])
    return summary, per_class

def eval_world_coco(model, desc="COCO"):
    """Zero-shot COCO evaluation of a model whose classes are the 80 COCO prompts in the normal order."""
    G = coco_gt()
    preds = predict_records(model, coco_records, desc)
    return coco_evaluate(to_coco_detections(preds, G["file_to_id"], G["idx_to_cat"]), COCO_IMG_IDS)

COCO_IMG_IDS = [coco_gt()["file_to_id"][Path(r["image_path"]).name] for r in coco_records]


# ---------- LVIS (official lvis evaluation) ----------
_LVIS = {}
def lvis_gt():
    if not _LVIS:
        # The lvis package still uses np.float and similar old names that NumPy 2 removed
        for name, typ in [("float", float), ("int", int), ("bool", bool), ("object", object)]:
            if name not in np.__dict__:
                setattr(np, name, typ)
        from lvis import LVIS
        with quiet():
            gt = LVIS(str(LVIS_DIR / "lvis_v1_val.json"))
        _LVIS.update(gt=gt,
                     file_to_id={Path(im["coco_url"]).name: im["id"] for im in gt.dataset["images"]},
                     idx_to_cat=sorted(gt.get_cat_ids()))
    return _LVIS

def lvis_evaluate(dets, img_ids):
    """Official LVIS box evaluation on our subset: AP, AP50, AP75, APs/m/l and APr/APc/APf (in %)."""
    from lvis import LVISResults, LVISEval
    G = lvis_gt()
    with quiet():
        E = LVISEval(G["gt"], LVISResults(G["gt"], dets, max_dets=CFG["MAX_DET"]), "bbox")
        E.params.img_ids = sorted(img_ids)
        E.run()
    return {k: (v * 100 if v >= 0 else np.nan) for k, v in E.get_results().items()}


# ---------- Custom dataset (Ultralytics validator) ----------
def export_custom_splits(splits, names, out_dir):
    """Save train/val/test in the YOLO folder layout and write a data.yaml with absolute paths for Ultralytics."""
    out_dir = Path(out_dir).resolve()
    for split, recs in splits.items():
        if recs:
            export_yolo_format(recs, names, out_dir, split=split)      # from Section 7d
    cfg = {"path": str(out_dir), "train": "images/train", "val": "images/val",
           "test": "images/test" if splits.get("test") else "images/val",
           "names": {i: n for i, n in enumerate(names)}}
    with open(out_dir / "data.yaml", "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out_dir / "data.yaml"

custom_yaml = None
if custom_train:
    custom_yaml = export_custom_splits({"train": custom_train, "val": custom_val, "test": custom_test},
                                       custom_names, CFG["ROOT"] / "custom_yolo")
    print("custom data for Ultralytics:", custom_yaml)

def ultra_val(model, data_yaml, split, name):
    """Ultralytics evaluation. Returns the summary (in %) and AP50-95 per class name (in %)."""
    m = model.val(data=str(data_yaml), split=split, imgsz=CFG["IMG_SIZE"], batch=CFG["BATCH_SIZE"],
                  conf=CFG["CONF_EVAL"], iou=0.7, device=ULTRA_DEVICE, workers=CFG["NUM_WORKERS"],
                  plots=False, verbose=False, project=str(RUNS_DIR / "val"), name=name, exist_ok=True)
    summary = dict(mAP50_95=m.box.map * 100, mAP50=m.box.map50 * 100, mAP75=m.box.map75 * 100,
                   precision=m.box.mp * 100, recall=m.box.mr * 100)
    per_class = {custom_names[int(c)]: float(ap) * 100 for c, ap in zip(m.ap_class_index, m.box.ap)}
    return summary, per_class


# ---------- Speed ----------
def measure_speed(model, images, warmup=10, half=False):
    """Time the model on images that are already in memory (BGR arrays), batch size 1.
    Ultralytics measures pre-processing, inference and post-processing separately (r.speed, in ms)."""
    kw = dict(imgsz=CFG["IMG_SIZE"], conf=CFG["CONF_SPEED"], device=ULTRA_DEVICE, verbose=False)
    if half:
        kw["half"] = True
    for im in (images * warmup)[:warmup]:
        model.predict(im, **kw)
    if DEVICE == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    pre, inf, post, wall = [], [], [], []
    for im in images:
        t0 = time.perf_counter()
        r = model.predict(im, **kw)[0]
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        wall.append((time.perf_counter() - t0) * 1000)
        pre.append(r.speed["preprocess"]); inf.append(r.speed["inference"]); post.append(r.speed["postprocess"])
    return dict(pre_ms=np.mean(pre), infer_ms=np.mean(inf), post_ms=np.mean(post), total_ms=np.mean(wall),
                fps_model=1000 / np.mean(inf), fps_end_to_end=1000 / np.mean(wall),
                peak_gpu_mem_MB=torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else np.nan)

print("helpers ready | COCO subset images:", len(COCO_IMG_IDS))

## 9. Zero-shot evaluation (next step 2)

### 9a. COCO val2017 subset

The pretrained **YOLO-World-S** (v2) gets the 80 COCO class names as prompts (plain template `"{}"`, the same texts as the official `coco_class_texts.json`, see Section 6b). It was **not trained on COCO boxes**, so this is a zero-shot test. As written in our report, many COCO classes are also in the pre-training data, so this tests general quality more than truly unseen classes.

We report the metrics planned in the report: mAP@[.5:.95], AP50, AP75, AP for small, medium and large objects, AR@100 and AP per class.

**Note:** these numbers are for our 500-image subset, not the full 5,000 images, so they can differ a little from published full-split numbers.

In [ ]:
world_s = load_world("s", build_prompts(coco_names, "plain"))
t0 = time.time()
coco_zs, coco_zs_cls = eval_world_coco(world_s, "COCO zero-shot (S)")
print(f"evaluated {len(coco_records)} images in {time.time() - t0:.0f}s")

RESULTS["coco_zero_shot"] = pd.DataFrame([dict(model="YOLO-World-S v2", prompts="plain", **coco_zs)]).round(2)
display(RESULTS["coco_zero_shot"])

# AP per class, with how often the class appears in the full val2017 split (from the EDA in Section 2c)
full_count = class_counts.set_index("category_id")["count"].reindex(coco_gt()["idx_to_cat"]).fillna(0).values
sub_count = np.bincount(np.concatenate([r["labels"] for r in coco_records]), minlength=len(coco_names))
coco_cls = pd.DataFrame({"class": coco_names, "AP": coco_zs_cls, "boxes_in_subset": sub_count,
                         "boxes_in_val2017": full_count.astype(int)})
RESULTS["coco_per_class"] = coco_cls.round(2)
ranked = coco_cls.dropna(subset=["AP"]).sort_values("AP", ascending=False)
print("best 10 classes:");  display(ranked.head(10).round(1).reset_index(drop=True))
print("worst 10 classes:"); display(ranked.tail(10).round(1).reset_index(drop=True))
rho = coco_cls[["AP", "boxes_in_val2017"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation between class AP and class frequency: {rho:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(["small", "medium", "large"], [coco_zs["APs"], coco_zs["APm"], coco_zs["APl"]], color=["#d62728", "#ff7f0e", "#2ca02c"])
ax[0].set_ylabel("AP (%)"); ax[0].set_title("Zero-shot COCO AP by object size (YOLO-World-S)")
ax[1].scatter(coco_cls["boxes_in_val2017"], coco_cls["AP"], s=14)
for _, r in ranked.head(3).iterrows():
    ax[1].annotate(r["class"], (r["boxes_in_val2017"], r["AP"]), fontsize=8)
for _, r in ranked.tail(3).iterrows():
    ax[1].annotate(r["class"], (r["boxes_in_val2017"], r["AP"]), fontsize=8)
ax[1].set_xscale("log"); ax[1].set_xlabel("boxes of the class in val2017 (log)"); ax[1].set_ylabel("AP (%)")
ax[1].set_title(f"AP per class vs class frequency (Spearman {rho:.2f})")
plt.tight_layout(); plt.show()

### 9b. LVIS v1 val subset

LVIS has 1,203 classes. The paper's main zero-shot result is on LVIS, and the classes are split into **rare** (1-10 training images), **common** (11-100) and **frequent** (>100). We give all 1,203 names as prompts and use the official LVIS evaluation, which also handles LVIS's *federated* labels (a class is only scored on images where it was checked).

On only 500 images many rare classes do not appear at all, so APr is based on few classes and is noisy. We print how many classes of each group are in our subset.

In [ ]:
if CFG["USE_LVIS"] and lvis_records:
    try:
        LG = lvis_gt()
        world_lvis = load_world("s", build_prompts(lvis_names, "plain"))
        preds = predict_records(world_lvis, lvis_records, "LVIS zero-shot (S)")
        dets = to_coco_detections(preds, LG["file_to_id"], LG["idx_to_cat"])
        lvis_ids = [LG["file_to_id"][Path(r["image_path"]).name] for r in lvis_records]
        lvis_res = lvis_evaluate(dets, lvis_ids)
        RESULTS["lvis_zero_shot"] = pd.DataFrame([dict(model="YOLO-World-S v2", prompts="plain", **lvis_res)]).round(2)
        display(RESULTS["lvis_zero_shot"])

        present = {LG["idx_to_cat"][c] for r in lvis_records for c in r["labels"].tolist()}
        freq = pd.Series([LG["gt"].cats[c]["frequency"] for c in present]).value_counts()
        print("classes with boxes in our subset: rare", int(freq.get("r", 0)), "| common", int(freq.get("c", 0)),
              "| frequent", int(freq.get("f", 0)))
        del world_lvis; free_memory()
    except Exception as e:
        print("LVIS evaluation did not run:", repr(e))
else:
    print("LVIS skipped (CFG['USE_LVIS'] = False)")

## 10. Prompt sensitivity (next step 3, part a)

**Research question.** How sensitive is zero-shot YOLO-World to the wording of the class prompts, and does a better wording help the classes with awkward names?

In YOLO-World a class score is the similarity between a region and the **text embedding of the prompt**, so the wording may change the result. The paper does not study this closely. We keep the model and images fixed and change only the prompts:

| Variant | Prompt for "dining table" | Idea |
|---|---|---|
| `plain` | `dining table` | the official COCO texts (baseline) |
| `photo` | `a photo of a dining table` | the CLIP-style sentence |
| `object` | `a dining table object` | a different sentence |
| `synonyms` | `table` | plain, but 8 awkward COCO names are replaced by more common words |

For every variant we also measure how far its CLIP embeddings move from the plain ones (mean cosine similarity, with our own text encoder from Section 6). In 10b we do the same on the custom dataset, where more exact animal names can be tested.

### 10a. COCO

In [ ]:
# More common words for some COCO names (the report mentions "hair drier" and "dining table")
COCO_PROMPT_OVERRIDES = {
    "hair drier": "hair dryer", "dining table": "table", "potted plant": "house plant", "tv": "television",
    "cell phone": "mobile phone", "sports ball": "ball", "mouse": "computer mouse", "remote": "remote control",
}
assert set(COCO_PROMPT_OVERRIDES) <= set(coco_names), "an override key is not a COCO class name"
PROMPT_VARIANTS = {"plain": ("plain", None), "photo": ("photo", None), "object": ("object", None),
                   "synonyms": ("plain", COCO_PROMPT_OVERRIDES)}

rows, prompt_cls = [], {}
for vname, (template, overrides) in PROMPT_VARIANTS.items():
    prompts = build_prompts(coco_names, template, overrides)
    if vname == "plain":
        summary, per_cls = coco_zs, coco_zs_cls                  # measured in Section 9a
    else:
        m = load_world("s", prompts)
        summary, per_cls = eval_world_coco(m, f"COCO prompts: {vname}")
        del m; free_memory()
    cos = (encode_texts(prompts) * coco_text_emb).sum(-1).mean().item()   # similarity to the plain embeddings
    prompt_cls[vname] = per_cls
    rows.append(dict(variant=vname, example=prompts[coco_names.index("dining table")],
                     text_cosine_to_plain=cos, **summary))

prompt_df = pd.DataFrame(rows)
prompt_df["dAP_vs_plain"] = prompt_df["AP"] - prompt_df.loc[prompt_df["variant"] == "plain", "AP"].item()
RESULTS["prompt_sensitivity_coco"] = prompt_df.round(3)
display(RESULTS["prompt_sensitivity_coco"])

# Classes whose name we replaced: AP with the plain name vs AP with the new word
idx = [coco_names.index(n) for n in COCO_PROMPT_OVERRIDES]
syn = pd.DataFrame({"class": list(COCO_PROMPT_OVERRIDES), "new prompt": list(COCO_PROMPT_OVERRIDES.values()),
                    "boxes_in_subset": sub_count[idx], "AP plain": prompt_cls["plain"][idx],
                    "AP synonym": prompt_cls["synonyms"][idx]})
syn["change"] = syn["AP synonym"] - syn["AP plain"]
RESULTS["prompt_synonyms_per_class"] = syn.round(2)
display(RESULTS["prompt_synonyms_per_class"])

# The 10 classes that change most when the template changes from plain to "a photo of a ..."
d = pd.DataFrame({"class": coco_names, "AP plain": prompt_cls["plain"], "AP photo": prompt_cls["photo"]})
d["change"] = d["AP photo"] - d["AP plain"]
print("largest per-class changes, plain -> photo template:")
display(d.reindex(d["change"].abs().sort_values(ascending=False).index).head(10).round(1).reset_index(drop=True))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].bar(prompt_df["variant"], prompt_df["AP"]); ax[0].set_ylabel("COCO AP (%)")
ax[0].set_ylim(max(0, prompt_df["AP"].min() - 5), prompt_df["AP"].max() + 2)
ax[0].set_title("Zero-shot COCO AP for each prompt variant (S)")
ax[1].barh(syn["class"] + " -> " + syn["new prompt"], syn["change"],
           color=["#2ca02c" if v >= 0 else "#d62728" for v in syn["change"].fillna(0)])
ax[1].axvline(0, color="k", lw=0.8); ax[1].set_xlabel("AP change (points)"); ax[1].set_title("Effect of the synonym")
plt.tight_layout(); plt.show()

### 10b. Custom dataset (zero-shot, test split)

Before any fine tuning, YOLO-World-S gets the custom class names as prompts. We compare the plain names, the CLIP sentence and more exact names (for example `rhinoceros` instead of `rhino`). The `plain` result is also the **zero-shot baseline** for Section 12.

In [ ]:
CUSTOM_ZS = {}
if custom_yaml:
    overrides = {k: v for k, v in CUSTOM_DATASET.get("prompt_overrides", {}).items() if k in custom_names}
    CUSTOM_PROMPT_VARIANTS = {"plain": ("plain", None), "photo": ("photo", None)}
    if overrides:
        CUSTOM_PROMPT_VARIANTS["specific"] = ("plain", overrides)

    # Note: the Ultralytics validator sets the vocabulary of a YOLO-World model to the class names in
    # data.yaml, so set_classes() alone would be ignored here. We therefore write one data.yaml per variant
    # whose class names ARE the prompts (same images and labels, same class order).
    base_cfg = yaml.safe_load(open(custom_yaml))
    rows = []
    for vname, (template, ov) in CUSTOM_PROMPT_VARIANTS.items():
        prompts = build_prompts(custom_names, template, ov)
        variant_yaml = custom_yaml.parent / f"data_prompts_{vname}.yaml"
        with open(variant_yaml, "w") as f:
            yaml.safe_dump({**base_cfg, "names": dict(enumerate(prompts))}, f, sort_keys=False)
        m = load_world("s", prompts)
        summary, per_cls = ultra_val(m, variant_yaml, "test", f"zero_shot_{vname}")
        CUSTOM_ZS[vname] = (summary, per_cls)
        rows.append(dict(variant=vname, prompts=" | ".join(prompts), **summary,
                         **{f"AP {n}": per_cls.get(n, np.nan) for n in custom_names}))
        del m; free_memory()
    RESULTS["prompt_sensitivity_custom"] = pd.DataFrame(rows).round(2)
    display(RESULTS["prompt_sensitivity_custom"])
else:
    print("No custom dataset -> skipped.")

## 11. Speed and accuracy of the S, M and L models (next step 3, part b)

**Research question.** How much accuracy does each step up in model size (S → M → L) buy, and what does it cost in latency, memory and parameters on our hardware?

Speed is the main promise of YOLO-World, so for each released size (v2 weights) we measure on the same hardware:

* zero-shot COCO AP on our subset (same protocol as Section 9a),
* parameters (millions),
* time per image at batch size 1 and 640 px: pre-processing, model inference, post-processing (NMS), and the full `predict` call,
* FPS of the model alone and of the full call, and peak GPU memory,
* on a GPU also the inference time in half precision (FP16).

The vocabulary is the 80 COCO prompts, set once before timing (prompt then detect), so the text encoder is not part of the timed loop. The hardware is printed in `RUN_INFO`; the paper's 52 FPS is for a V100 GPU, so our absolute numbers will differ.

In [ ]:
SPEED_IMGS = [cv2.imread(r["image_path"]) for r in coco_records[:CFG["SPEED_IMAGES"]]]
rows = []
for size in CFG["MODEL_SIZES"]:
    m = world_s if size == "s" else load_world(size, build_prompts(coco_names, "plain"))
    summary = coco_zs if size == "s" else eval_world_coco(m, f"COCO zero-shot ({size.upper()})")[0]
    speed = measure_speed(m, SPEED_IMGS)
    row = dict(model=f"YOLO-World-{size.upper()} v2", size=size, params_M=count_params(m),
               AP=summary["AP"], AP50=summary["AP50"], AP75=summary["AP75"], APs=summary["APs"], **speed)
    if DEVICE == "cuda":
        row["infer_ms_fp16"] = measure_speed(m, SPEED_IMGS, half=True)["infer_ms"]
    rows.append(row)
    print(f"{row['model']}: AP {row['AP']:.1f} | inference {row['infer_ms']:.1f} ms | {row['fps_model']:.0f} FPS (model)")
    if size != "s":
        del m; free_memory()

size_df = pd.DataFrame(rows)
RESULTS["model_size_speed"] = size_df.round(2)
display(RESULTS["model_size_speed"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(size_df["infer_ms"], size_df["AP"], "o-")
for _, r in size_df.iterrows():
    ax[0].annotate(r["model"].split()[0], (r["infer_ms"], r["AP"]), textcoords="offset points", xytext=(5, -10))
ax[0].set_xlabel(f"inference time per image (ms) on {RUN_INFO['hardware']}"); ax[0].set_ylabel("COCO AP (%)")
ax[0].set_title("Speed vs accuracy (zero-shot)")
ax[1].bar(size_df["model"].str.split().str[0], size_df["params_M"]); ax[1].set_ylabel("parameters (M)")
ax[1].set_title("Model size")
plt.tight_layout(); plt.show()

### 11b. Reproduction: our numbers vs. the reported numbers

Here we put our measured numbers next to the **reported** ones. The reported numbers are copied from the sources and are *not* our results:

* **Paper, Table 2:** zero-shot LVIS *minival*, Fixed AP, FPS on one V100 without TensorRT, **v1** weights.
* **Ultralytics docs:** zero-shot COCO val2017 (mAP50-95, mAP50, mAP75) for the **v2** weights that we use.

The settings are not identical, so a gap is expected even with a perfect reproduction:

| Difference | Effect on the comparison |
|---|---|
| 500-image subsets instead of the full splits | random noise of a few AP points |
| LVIS v1 *val* subset instead of *minival*; standard AP instead of *Fixed AP* | Fixed AP removes the 300-boxes-per-image limit per class and is usually a little higher |
| v2 weights (us) vs v1 weights (paper Table 2) | v2 changes the neck design, so AP differs slightly |
| T4 GPU (Colab) vs V100 GPU; Ultralytics predict vs the paper's timing script | absolute FPS is not comparable, only the S < M < L trend is |

In [ ]:
# Reported numbers (copied from the sources; not our results)
PAPER_LVIS_TABLE2 = pd.DataFrame([   # Cheng et al., CVPR 2024, Table 2 (zero-shot LVIS minival, Fixed AP, V100)
    dict(size="s", paper_params_M=13, paper_fps_v100=74.1, paper_AP=26.2, paper_APr=19.1, paper_APc=23.6, paper_APf=29.8),
    dict(size="m", paper_params_M=29, paper_fps_v100=58.1, paper_AP=31.0, paper_APr=23.8, paper_APc=29.2, paper_APf=33.9),
    dict(size="l", paper_params_M=48, paper_fps_v100=52.0, paper_AP=35.0, paper_APr=27.1, paper_APc=32.8, paper_APf=38.3),
])
ULTRA_COCO = pd.DataFrame([           # docs.ultralytics.com/models/yolo-world, zero-shot COCO, v2 weights
    dict(size="s", rep_AP=37.7, rep_AP50=52.2, rep_AP75=41.0),
    dict(size="m", rep_AP=43.0, rep_AP50=58.4, rep_AP75=46.8),
    dict(size="l", rep_AP=45.8, rep_AP50=61.3, rep_AP75=49.8),
])

# COCO zero-shot: ours (subset) vs reported (full val2017)
coco_cmp = size_df[["model", "size", "AP", "AP50", "AP75"]].merge(ULTRA_COCO, on="size", how="left")
coco_cmp["AP_gap"] = coco_cmp["AP"] - coco_cmp["rep_AP"]
coco_cmp = coco_cmp.rename(columns={"AP": "our_AP", "AP50": "our_AP50", "AP75": "our_AP75"})
RESULTS["reproduction_coco"] = coco_cmp.drop(columns="size").round(2)
print(f"Zero-shot COCO: ours on {len(coco_records)} images vs. reported on the full val2017 (Ultralytics docs)")
display(RESULTS["reproduction_coco"])

# Speed and size: ours vs paper (different GPU, see the table above)
spd_cmp = size_df[["model", "size", "params_M", "infer_ms", "fps_model"]].merge(
    PAPER_LVIS_TABLE2[["size", "paper_params_M", "paper_fps_v100"]], on="size", how="left")
spd_cmp["hardware"] = RUN_INFO["hardware"]
RESULTS["reproduction_speed"] = spd_cmp.drop(columns="size").round(2)
print("Speed and size: ours vs. paper Table 2 (V100, re-parameterized model)")
display(RESULTS["reproduction_speed"])

# LVIS zero-shot (S model): ours vs paper Table 2
if "lvis_zero_shot" in RESULTS:
    ours = RESULTS["lvis_zero_shot"].iloc[0]
    p = PAPER_LVIS_TABLE2.iloc[0]
    lvis_cmp = pd.DataFrame([dict(metric=k, ours_v2_val_subset=ours[k], paper_v1_minival=p[f"paper_{k}"],
                                  gap=ours[k] - p[f"paper_{k}"]) for k in ["AP", "APr", "APc", "APf"]])
    RESULTS["reproduction_lvis"] = lvis_cmp.round(2)
    print(f"Zero-shot LVIS, YOLO-World-S: ours ({len(lvis_records)} val images) vs. paper Table 2 (minival)")
    display(RESULTS["reproduction_lvis"])

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(len(coco_cmp)); w = 0.38
ax.bar(x - w / 2, coco_cmp["our_AP"], w, label=f"ours ({len(coco_records)}-image subset)")
ax.bar(x + w / 2, coco_cmp["rep_AP"], w, label="reported (Ultralytics, full val2017)")
ax.set_xticks(x); ax.set_xticklabels(coco_cmp["model"]); ax.set_ylabel("zero-shot COCO AP (%)")
ax.set_title("Reproduction: zero-shot COCO AP"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 12. Additional experiment: fine-tuned YOLO-World vs. a closed-set YOLOv8 (next step 4)

**Research question.** On a small custom dataset, does fine-tuning the open-vocabulary YOLO-World-S give better accuracy than training a normal closed-set YOLOv8-S of the same size? Does its advantage depend on whether a class is in COCO's 80 classes? And how much of its open-vocabulary ability (COCO AP) does it lose by fine-tuning? This is a **baseline comparison** (Deliverable 3, §2.4). Sections 10 and 11 add an ablation (prompt wording) and a computational analysis (model size).

We train two models on the **same** custom training split with the **same** settings (epochs, 640 px, batch size, seed and the default YOLOv8 augmentations, which are the steps of Section 4: mosaic, random zoom and shift, HSV colour change, flip; mosaic is turned off in the last 10 epochs):

| Model | Start weights | Classes |
|---|---|---|
| **YOLO-World-S fine-tuned** | `yolov8s-worldv2.pt` (open-vocabulary pre-training) | text prompts of the 4 class names |
| **YOLOv8-S** | `yolov8s.pt` (COCO pre-training) | a normal fixed 4-class head (closed set) |

**Optimizer recipe.** Each model uses its own standard recipe. YOLOv8-S uses the Ultralytics defaults. YOLO-World-S uses the settings of the **official YOLO-World fine-tuning config** (`yolo_world_v2_s_..._finetune_coco.py`: AdamW, learning rate 2e-4, weight decay 0.05), with the bias warm-up turned off. We tested the Ultralytics defaults for YOLO-World first (AdamW, learning rate 0.00125, biases warmed up from learning rate 0.1). In a short CPU test (5 epochs, 10% of the training images) they made it *worse* than zero-shot: test mAP50-95 fell from 56.3 to 33.5, and the val mAP kept falling during warm-up. The high warm-up learning rate damages the pretrained text-matching head, whose bias and logit-scale parameters are in the bias group. With the official recipe, the same short test gave test mAP50-95 56.2 and stable val mAP (52.4 → 53.6 → 44.7 over 5 epochs). This is a feasibility adjustment, not part of the experiment.

Ultralytics picks the best epoch on the *val* split. We report the final numbers on the *test* split, which neither model saw. The **zero-shot** YOLO-World-S from Section 10b (no training) is the third row of the comparison.

**Forgetting check.** After fine tuning we give the fine-tuned YOLO-World the 80 COCO names again and measure COCO AP. A drop compared with Section 9a shows how much of the open-vocabulary ability is lost by fine tuning on 4 classes.

In [ ]:
FT = {}
if custom_yaml:
    TRAIN_ARGS = dict(data=str(custom_yaml), epochs=CFG["EPOCHS"], imgsz=CFG["IMG_SIZE"], batch=CFG["TRAIN_BATCH"],
                      seed=SEED, deterministic=True, device=ULTRA_DEVICE, workers=CFG["NUM_WORKERS"],
                      project=str(RUNS_DIR / "train"), exist_ok=True, plots=True, verbose=False)
    # Optimizer recipe of each model (everything else above is shared):
    # YOLO-World uses the official fine-tuning config (AdamW, lr 2e-4, weight decay 0.05) with no bias warm-up;
    # YOLOv8 uses the standard Ultralytics recipe. See the text above for why.
    RECIPES = {"yolo_world_s": dict(optimizer="AdamW", lr0=2e-4, weight_decay=0.05, warmup_bias_lr=0.0),
               "yolov8s": dict()}
    RUN_INFO["train_recipes"] = RECIPES
    for kind in ["yolo_world_s", "yolov8s"]:
        model = YOLOWorld("yolov8s-worldv2.pt") if kind == "yolo_world_s" else YOLO("yolov8s.pt")
        t0 = time.time()
        model.train(name=kind, **TRAIN_ARGS, **RECIPES[kind])
        FT[kind] = dict(best=Path(model.trainer.best), save_dir=Path(model.trainer.save_dir),
                        train_min=(time.time() - t0) / 60)
        print(f"{kind}: trained in {FT[kind]['train_min']:.1f} min -> {FT[kind]['best']}")
        del model; free_memory()

    # Copy the trained weights to weights/ (the deliverable and what app.py loads) and some held-out
    # TEST images to samples/ for the live demo
    import shutil
    Path("weights").mkdir(exist_ok=True); Path("samples").mkdir(exist_ok=True)
    for kind in FT:
        shutil.copy(FT[kind]["best"], Path("weights") / f"{kind}_custom.pt")
    for r in random.Random(SEED).sample(custom_test, min(8, len(custom_test))):
        shutil.copy(r["image_path"], Path("samples") / f"test_{Path(r['image_path']).name}")
    print("trained weights:", sorted(p.name for p in Path("weights").glob("*_custom.pt")),
          "| demo samples:", len(list(Path("samples").glob("*"))))
else:
    print("No custom dataset -> fine tuning skipped.")

In [ ]:
if FT:
    CUSTOM_SPEED_IMGS = [cv2.imread(r["image_path"]) for r in custom_test[:CFG["SPEED_IMAGES"]]]
    zs_summary, zs_cls = CUSTOM_ZS["plain"]
    zs_model = load_world("s", build_prompts(custom_names, "plain"))
    rows = [dict(model="YOLO-World-S zero-shot", train_min=0.0, params_M=count_params(zs_model), **zs_summary,
                 infer_ms=measure_speed(zs_model, CUSTOM_SPEED_IMGS)["infer_ms"],
                 **{f"AP {n}": zs_cls.get(n, np.nan) for n in custom_names})]
    del zs_model; free_memory()

    for kind, label in [("yolo_world_s", "YOLO-World-S fine-tuned"), ("yolov8s", "YOLOv8-S fine-tuned (closed set)")]:
        m = YOLOWorld(str(FT[kind]["best"])) if kind == "yolo_world_s" else YOLO(str(FT[kind]["best"]))
        summary, per_cls = ultra_val(m, custom_yaml, "test", f"test_{kind}")
        rows.append(dict(model=label, train_min=FT[kind]["train_min"], params_M=count_params(m), **summary,
                         infer_ms=measure_speed(m, CUSTOM_SPEED_IMGS)["infer_ms"],
                         **{f"AP {n}": per_cls.get(n, np.nan) for n in custom_names}))
        del m; free_memory()

    ft_df = pd.DataFrame(rows)
    RESULTS["finetune_comparison"] = ft_df.round(2)
    display(RESULTS["finetune_comparison"])

    # AP per class, grouped by "in COCO 80" or not
    in_coco = {n: n in coco_names for n in custom_names}
    per_cls_df = ft_df.set_index("model")[[f"AP {n}" for n in custom_names]].T
    per_cls_df.index = [f"{n} ({'in COCO' if in_coco[n] else 'outside COCO'})" for n in custom_names]
    RESULTS["finetune_per_class"] = per_cls_df.round(2).reset_index(names="class")

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    per_cls_df.plot.bar(ax=ax[0], rot=0); ax[0].set_ylabel("AP50-95 on test (%)")
    ax[0].set_title("AP per class on the custom test split"); ax[0].legend(fontsize=8)
    for kind, label in [("yolo_world_s", "YOLO-World-S"), ("yolov8s", "YOLOv8-S")]:
        log = pd.read_csv(FT[kind]["save_dir"] / "results.csv")
        log.columns = log.columns.str.strip()
        ax[1].plot(log["epoch"], log["metrics/mAP50-95(B)"] * 100, label=label)
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val mAP50-95 (%)"); ax[1].set_title("Training curves (val split)")
    ax[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Forgetting check: does the fine-tuned YOLO-World still detect the 80 COCO classes?
if FT:
    try:
        m = YOLOWorld(str(FT["yolo_world_s"]["best"]))
        m.set_classes(build_prompts(coco_names, "plain"))
        after, _ = eval_world_coco(m, "COCO after fine tuning")
        forget = pd.DataFrame([dict(model="YOLO-World-S before fine tuning", **coco_zs),
                               dict(model="YOLO-World-S after fine tuning", **after)])
        RESULTS["forgetting_coco"] = forget.round(2)
        display(RESULTS["forgetting_coco"])
        print(f"COCO AP change after fine tuning on {len(custom_names)} classes: {after['AP'] - coco_zs['AP']:+.1f} points")
        del m; free_memory()
    except Exception as e:
        print("forgetting check did not run:", repr(e))

## 13. Results summary

All tables are saved as CSV files in `data/results/`, together with `run_info.json` (versions, hardware and settings) and `summary.md` (all tables in one file for the report). The key findings below are written by the code from the measured numbers.

In [ ]:
for key, df in RESULTS.items():
    df.to_csv(RESULTS_DIR / f"{key}.csv", index=False)
with open(RESULTS_DIR / "run_info.json", "w") as f:
    json.dump(RUN_INFO, f, indent=1)

def table_md(df):
    """Simple Markdown table (no extra library needed)."""
    cols = list(df.columns)
    fmt = lambda v: f"{v:.2f}" if isinstance(v, (float, np.floating)) else str(v)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

findings = []
if "coco_zero_shot" in RESULTS:
    findings.append(f"Zero-shot COCO ({len(coco_records)} images), YOLO-World-S: AP {coco_zs['AP']:.1f}, "
                    f"AP50 {coco_zs['AP50']:.1f}, AP small/medium/large {coco_zs['APs']:.1f} / {coco_zs['APm']:.1f} / {coco_zs['APl']:.1f}.")
if "reproduction_coco" in RESULTS:
    findings.append("Reproduction gap, zero-shot COCO AP (ours on the subset minus reported): " + "; ".join(
        f"{r['model'].split()[0]} {r['AP_gap']:+.1f}" for _, r in RESULTS["reproduction_coco"].iterrows()) + ".")
if "reproduction_lvis" in RESULTS:
    g = RESULTS["reproduction_lvis"].set_index("metric")["gap"]
    findings.append(f"Reproduction gap, zero-shot LVIS AP of the S model (ours minus paper Table 2): {g['AP']:+.1f}.")
if "lvis_zero_shot" in RESULTS:
    r = RESULTS["lvis_zero_shot"].iloc[0]
    findings.append(f"Zero-shot LVIS ({len(lvis_records)} images): AP {r['AP']:.1f}, APr {r['APr']:.1f}, "
                    f"APc {r['APc']:.1f}, APf {r['APf']:.1f}.")
if "prompt_sensitivity_coco" in RESULTS:
    p = RESULTS["prompt_sensitivity_coco"]
    best = p.loc[p["AP"].idxmax()]
    findings.append(f"Prompt wording on COCO: AP ranges from {p['AP'].min():.1f} to {p['AP'].max():.1f} "
                    f"(spread {p['AP'].max() - p['AP'].min():.1f} points); best variant: '{best['variant']}'.")
if "prompt_sensitivity_custom" in RESULTS:
    p = RESULTS["prompt_sensitivity_custom"]
    best = p.loc[p["mAP50_95"].idxmax()]
    findings.append(f"Prompt wording on the custom test split: mAP50-95 from {p['mAP50_95'].min():.1f} to "
                    f"{p['mAP50_95'].max():.1f}; best variant: '{best['variant']}'.")
if "model_size_speed" in RESULTS:
    s = RESULTS["model_size_speed"]
    findings.append("S/M/L on " + RUN_INFO["hardware"] + ": " + "; ".join(
        f"{r['model'].split()[0]} AP {r['AP']:.1f} at {r['infer_ms']:.1f} ms ({r['fps_model']:.0f} FPS)"
        for _, r in s.iterrows()) + ".")
if "finetune_comparison" in RESULTS:
    f_ = RESULTS["finetune_comparison"]
    findings.append("Custom test split mAP50-95: " + "; ".join(
        f"{r['model']} {r['mAP50_95']:.1f}" for _, r in f_.iterrows()) + ".")
if "forgetting_coco" in RESULTS:
    fg = RESULTS["forgetting_coco"]["AP"].tolist()
    findings.append(f"COCO AP of YOLO-World-S before / after fine tuning: {fg[0]:.1f} / {fg[1]:.1f}.")

summary_md = ["# YOLO-World experiments: results", "",
              "Run info: " + ", ".join(f"{k}={v}" for k, v in RUN_INFO.items()), "", "## Key findings", ""]
summary_md += [f"- {x}" for x in findings]
for key, df in RESULTS.items():
    summary_md += ["", f"## {key}", "", table_md(df)]
(RESULTS_DIR / "summary.md").write_text("\n".join(summary_md), encoding="utf-8")

if CFG["QUICK_TEST"]:
    print("QUICK_TEST run: the numbers below only show that the code works. They are NOT results.\n")
print("Key findings")
for x in findings:
    print(" -", x)
print("\nsaved:", sorted(p.name for p in RESULTS_DIR.iterdir()))

### Limits of these experiments

* **Subsets.** COCO and LVIS use 500 images each (fixed seed), so the numbers are close to, but not the same as, full-split numbers. On LVIS, the rare-class AP rests on only a few classes.
* **One training run per model.** We use one seed. Small differences between the fine-tuned models (about 1 AP point or less) may be noise.
* **"Outside COCO" is not "unseen".** Buffalo and rhino are not in COCO's 80 classes, but they may be in YOLO-World's much larger pre-training data.
* **Speed depends on the hardware.** We report the GPU in `run_info.json`; only numbers from the same GPU should be compared.